# JWCM v2: Soft Attribution Merge (Qwen3-1.7B, Fisher Validation Reuse)

This notebook implements a Fisher-artifact-reuse JWCM v2 pipeline for IF/Math task merging:

1. Reuse Fisher validation splits from `/Qwen3-1.7B-fisher-merge/validation`.
2. Reuse Fisher pipeline correct rollouts when available, then compute token-level `Δlog p = log p_rl - log p_base`.
3. Identify critical tokens and save delta-logp statistics for analysis.
4. Compute JWCM importance scores:
   - Exact mode: `S_j = Σ |∂log p(y_t)/∂θ_j · Δθ_j|`
   - Approx mode: one-backward-per-sequence approximation for speed.
5. Save two merge ablations:
   - `jwcm_v2_soft`
   - `jwcm_v2_soft_sparsified`
6. Run sparsity/layer-distribution analysis for importance tensors.

All artifacts are saved under `/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-jwcm-v2-new`.


## Mathematical Setup

- Critical tokens for task `τ`:
  - `C_τ = {(x, y_t) | Δlog p_τ(y_t|x) > ε}`
- Importance score:
  - `S_{τ,j} = Σ_{(x,y_t) in C_τ} | ∂log p(y_t|x)/∂θ_j · Δθ_{τ,j} |`
- JWCM v2 soft merge:
  - `θ_merge,j = θ_base,j + Σ_τ(S_{τ,j}·Δθ_{τ,j}) / (Σ_τ S_{τ,j} + eps)`
- Sparsified variant:
  - `θ_merge,j = θ_base,j + 1[max_τ S_{τ,j} > κ] · Σ_τ(S_{τ,j}·Δθ_{τ,j}) / (Σ_τ S_{τ,j} + eps)`

Implementation notes:
- Validation size defaults to **512 per task** (total 1024), which is a pragmatic balance between signal quality and runtime.
- Critical tokens use top-`p` and minimum-`epsilon` jointly.


In [ ]:
from __future__ import annotations

import gc
import json
import random
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Mapping, MutableMapping, Sequence

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


@dataclass(frozen=True)
class TaskSpec:
    """Task-specific paths used by JWCM v2.

    Args:
        name: Short task name used in artifact naming.
        model_path: Local checkpoint path for the RL-tuned model.
        fisher_validation_path: Validation parquet reused from Fisher merging.
        fisher_correct_rollout_path: Optional path to Fisher-pipeline correct rollout parquet.
        data_path: Optional original train parquet path (kept for compatibility helpers).
        validation_samples: Optional sample count metadata for compatibility helpers.
    """

    name: str
    model_path: Path
    fisher_validation_path: Path
    fisher_correct_rollout_path: Path | None = None
    data_path: Path | None = None
    validation_samples: int = 512


@dataclass(frozen=True)
class CriticalTokenConfig:
    """Configuration for critical token extraction.

    Args:
        epsilon: Minimum `Δlog p` threshold for token criticality.
        top_p: Top proportion of tokens kept globally by `Δlog p`.
        max_prompt_tokens: Prompt truncation length before generation/evaluation.
        max_new_tokens: Maximum generated/evaluated continuation length.
        max_critical_tokens: Optional cap on total critical tokens per task.
    """

    epsilon: float = 0.05
    top_p: float = 0.05
    max_prompt_tokens: int = 1536
    max_new_tokens: int = 128
    max_critical_tokens: int | None = 4096


@dataclass(frozen=True)
class AttributionConfig:
    """Configuration for JWCM importance scoring.

    Args:
        mode: `exact_token_abs` for formula-faithful token-wise attribution,
            or `sequence_sum_approx` for a faster approximation.
        max_backprop_tokens: Optional cap for exact mode to bound runtime.
        normalize_by_token_count: Whether to divide final scores by processed token count.
    """

    mode: str = "exact_token_abs"
    max_backprop_tokens: int | None = 2048
    normalize_by_token_count: bool = True


@dataclass(frozen=True)
class MergeConfig:
    """Configuration for JWCM v2 merge variants.

    Args:
        epsilon: Numerical stabilizer in denominator (`ΣS + epsilon`).
        sparsify_mode: `absolute` uses fixed kappa; `sampled_quantile` estimates kappa.
        sparsify_kappa: Absolute threshold when `sparsify_mode=absolute`.
        sparsify_quantile: Quantile used when `sparsify_mode=sampled_quantile`.
        sampled_values_per_tensor: Per-parameter sample count for quantile estimation.
    """

    epsilon: float = 1e-8
    sparsify_mode: str = "sampled_quantile"
    sparsify_kappa: float = 0.0
    sparsify_quantile: float = 0.90
    sampled_values_per_tensor: int = 4096


@dataclass(frozen=True)
class TokenDeltaSourceConfig:
    """Priority and fallback policy for token-delta construction.

    Args:
        prefer_fisher_correct_rollout: Reuse Fisher-pipeline correct rollouts first.
        fallback_to_precomputed: Reuse previous JWCM precompute CSV/PT artifacts if needed.
        fallback_to_online_generation: Last-resort on-the-fly RL generation from validation prompts.
        use_only_correct_rows: Keep only `is_correct=True` rows when that column exists.
        max_rollouts_per_sample: Optional cap for rollout rows per validation sample.
    """

    prefer_fisher_correct_rollout: bool = True
    fallback_to_precomputed: bool = True
    fallback_to_online_generation: bool = True
    use_only_correct_rows: bool = True
    max_rollouts_per_sample: int | None = None


@dataclass(frozen=True)
class RuntimeConfig:
    """Runtime and path configuration for the notebook.

    Args:
        base_model_id: Base model identifier/path used as merge anchor.
        output_root: Directory where all artifacts will be written.
        seed: Global random seed for reproducibility.
        model_dtype: Torch dtype used for model loading.
        device: Execution device string (`cuda` or `cpu`).
    """

    base_model_id: str
    output_root: Path
    seed: int
    model_dtype: torch.dtype
    device: str


BASE_MODEL_ID = "Qwen/Qwen3-1.7B"
IF_MODEL_PATH = Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface")
MATH_MODEL_PATH = Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface")

FISHER_VALIDATION_ROOT = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/validation"
)
FISHER_PIPELINE_TASK_ROOT = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/qwen3_1.7b_fisher_pipeline_verbose/tasks"
)

LEGACY_PRECOMPUTED_TOKEN_ROOT = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-jwcm-v2/distributed_token_precompute"
)

TASK_SPECS: List[TaskSpec] = [
    TaskSpec(
        name="if",
        model_path=IF_MODEL_PATH,
        fisher_validation_path=FISHER_VALIDATION_ROOT / "if_validation.parquet",
        fisher_correct_rollout_path=FISHER_PIPELINE_TASK_ROOT / "if" / "correct_rollout_trajectories.parquet",
    ),
    TaskSpec(
        name="math",
        model_path=MATH_MODEL_PATH,
        fisher_validation_path=FISHER_VALIDATION_ROOT / "math_validation.parquet",
        fisher_correct_rollout_path=FISHER_PIPELINE_TASK_ROOT / "math" / "correct_rollout_trajectories.parquet",
    ),
]

RUNTIME = RuntimeConfig(
    base_model_id=BASE_MODEL_ID,
    output_root=Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-jwcm-v2-new"),
    seed=42,
    model_dtype=torch.bfloat16,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

CRITICAL_CFG = CriticalTokenConfig(
    epsilon=0.05,
    top_p=0.05,
    max_prompt_tokens=1536,
    max_new_tokens=128,
    max_critical_tokens=4096,
)

ATTR_CFG = AttributionConfig(
    mode="exact_token_abs",
    max_backprop_tokens=2048,
    normalize_by_token_count=True,
)

# ATTR_CFG = AttributionConfig(
#     mode="sequence_sum_approx",
#     max_backprop_tokens=2048,
#     normalize_by_token_count=True,
# )

MERGE_CFG = MergeConfig(
    epsilon=1e-12,
    sparsify_mode="sampled_quantile",
    sparsify_kappa=0.0,
    sparsify_quantile=0.90,
    sampled_values_per_tensor=4096,
)

TOKEN_SOURCE_CFG = TokenDeltaSourceConfig(
    prefer_fisher_correct_rollout=True,
    fallback_to_precomputed=True,
    fallback_to_online_generation=True,
    use_only_correct_rows=True,
    max_rollouts_per_sample=None,
)

# Keep validation prompt universe fully aligned with Fisher validation set.
# `None` means "use all rows from Fisher validation parquet".
ATTRIBUTION_VALIDATION_SAMPLES_PER_TASK: int | None = None

# Sample this many rows from the global correct-rollout pool per task.
# This is rollout-level sampling (trajectory-level), not prompt-level sampling.
ATTRIBUTION_CORRECT_ROLLOUT_SAMPLES_PER_TASK: int | None = 128

for required_path in [IF_MODEL_PATH, MATH_MODEL_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required model path does not exist: {required_path}")

for task_spec in TASK_SPECS:
    if not task_spec.fisher_validation_path.exists():
        raise FileNotFoundError(
            f"Missing Fisher validation parquet for task '{task_spec.name}': "
            f"{task_spec.fisher_validation_path}"
        )

RUNTIME.output_root.mkdir(parents=True, exist_ok=True)

print(f"Device: {RUNTIME.device}")
print(f"DType: {RUNTIME.model_dtype}")
print(f"Output root: {RUNTIME.output_root}")
print(f"Attribution mode: {ATTR_CFG.mode}")
print(f"Token-delta source config: {TOKEN_SOURCE_CFG}")
print(f"Legacy precomputed token root (fallback): {LEGACY_PRECOMPUTED_TOKEN_ROOT}")
if ATTRIBUTION_VALIDATION_SAMPLES_PER_TASK is None:
    print("Attribution validation samples per task: ALL rows from Fisher validation")
else:
    print(
        "Attribution validation samples per task: "
        f"{ATTRIBUTION_VALIDATION_SAMPLES_PER_TASK}"
    )

if ATTRIBUTION_CORRECT_ROLLOUT_SAMPLES_PER_TASK is None:
    print("Correct-rollout sampling per task: ALL available correct trajectories")
else:
    print(
        "Correct-rollout sampling per task (trajectory-level): "
        f"{ATTRIBUTION_CORRECT_ROLLOUT_SAMPLES_PER_TASK}"
    )

for task_spec in TASK_SPECS:
    rollout_path = task_spec.fisher_correct_rollout_path
    if rollout_path is None:
        continue
    if rollout_path.exists():
        print(f"Task={task_spec.name} Fisher correct rollout path: {rollout_path}")
    else:
        print(
            f"Task={task_spec.name} Fisher correct rollout path missing (fallbacks will be used): "
            f"{rollout_path}"
        )


In [ ]:
def set_seed(seed: int) -> None:
    """Set all relevant random seeds for reproducible sampling/generation.

    Args:
        seed: Integer random seed.

    Returns:
        None. Global RNG states are updated in-place.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def now_iso() -> str:
    """Return UTC timestamp string for metadata versioning.

    Returns:
        ISO-8601 timestamp string in UTC.
    """

    return datetime.utcnow().isoformat(timespec="seconds") + "Z"


def save_json(payload: Mapping[str, Any], output_path: Path) -> None:
    """Persist JSON payload with pretty formatting.

    Args:
        payload: JSON-serializable mapping object.
        output_path: Destination JSON path.

    Returns:
        None. Data is written to disk.
    """

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=False)


def normalize_prompt_messages(prompt_obj: Any) -> List[Dict[str, str]]:
    """Normalize dataset `prompt` field to chat-template-compatible messages.

    The IF/Math parquet files store `prompt` as `np.ndarray` containing dicts.
    This helper makes the representation consistent and validates required keys.

    Args:
        prompt_obj: Raw prompt object loaded from parquet.

    Returns:
        List of dictionaries with `role` and `content` keys.
    """

    if isinstance(prompt_obj, np.ndarray):
        messages = prompt_obj.tolist()
    elif isinstance(prompt_obj, list):
        messages = prompt_obj
    else:
        raise TypeError(f"Unsupported prompt type: {type(prompt_obj)}")

    normalized: List[Dict[str, str]] = []
    for message in messages:
        if not isinstance(message, dict):
            raise TypeError(f"Prompt message must be dict, got {type(message)}")
        role = str(message.get("role", "user"))
        content = str(message.get("content", ""))
        normalized.append({"role": role, "content": content})
    return normalized


def sample_validation_split(
    task_spec: TaskSpec,
    tokenizer: AutoTokenizer,
    seed: int,
    output_path: Path,
) -> pd.DataFrame:
    """Sample task validation prompts from training parquet and save to parquet.

    Args:
        task_spec: Task configuration with source parquet and sample size.
        tokenizer: Tokenizer used to render chat template into prompt text.
        seed: Deterministic seed used for row sampling.
        output_path: Destination parquet path for sampled validation data.

    Returns:
        DataFrame with rendered prompt text and provenance metadata.
    """

    raw_df = pd.read_parquet(task_spec.data_path, columns=["prompt"])
    sample_count = min(task_spec.validation_samples, len(raw_df))
    sampled_df = raw_df.sample(n=sample_count, random_state=seed, replace=False)

    records: List[Dict[str, Any]] = []
    for dataset_index, prompt_obj in zip(sampled_df.index.tolist(), sampled_df["prompt"].tolist()):
        messages = normalize_prompt_messages(prompt_obj)

        # Render prompt text exactly as the model expects before generation.
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        records.append(
            {
                "task": task_spec.name,
                "dataset_index": int(dataset_index),
                "prompt_messages_json": json.dumps(messages, ensure_ascii=False),
                "prompt_text": prompt_text,
            }
        )

    validation_df = pd.DataFrame(records)
    validation_df.insert(0, "sample_id", np.arange(len(validation_df), dtype=np.int64))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    validation_df.to_parquet(output_path, index=False)
    return validation_df


def load_json(input_path: Path) -> Dict[str, Any]:
    """Load JSON payload from disk.

    Args:
        input_path: Source JSON path.

    Returns:
        Parsed dictionary payload.
    """

    with input_path.open("r", encoding="utf-8") as file:
        return json.load(file)


In [ ]:
def load_tokenizer_with_mistral_regex_fix(model_name_or_path: str) -> AutoTokenizer:
    """Load tokenizer with optional `fix_mistral_regex=True` compatibility flag.

    Args:
        model_name_or_path: Hugging Face model id or local path.

    Returns:
        Tokenizer instance with trust-remote-code enabled.
    """

    try:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
            fix_mistral_regex=True,
        )
    except TypeError:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
        )


def load_causal_lm(
    model_name_or_path: str | Path,
    torch_dtype: torch.dtype,
    device: str,
) -> tuple[AutoModelForCausalLM, AutoTokenizer]:
    """Load causal LM and tokenizer with explicit device placement.

    Args:
        model_name_or_path: Hugging Face model id or local checkpoint path.
        torch_dtype: Tensor dtype for model weights.
        device: Target device string (`cpu` or `cuda`).

    Returns:
        Tuple of loaded `(model, tokenizer)`.
    """

    resolved_path = str(model_name_or_path)
    model = AutoModelForCausalLM.from_pretrained(
        resolved_path,
        torch_dtype=torch_dtype,
        device_map=None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model.to(device)
    model.eval()

    tokenizer = load_tokenizer_with_mistral_regex_fix(resolved_path)

    # Ensure generation can pad safely even for models without an explicit pad token.
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


@torch.no_grad()
def _compute_token_delta_rows_for_sequence(
    task_name: str,
    sample_id: int,
    sequence_id: int,
    dataset_index: int,
    full_ids: torch.Tensor,
    prompt_len: int,
    tokenizer: AutoTokenizer,
    base_model: AutoModelForCausalLM,
    rl_model: AutoModelForCausalLM,
    device: str,
) -> List[Dict[str, Any]]:
    """Compute per-token delta-logp rows for one prompt+response token sequence.

    Args:
        task_name: Task identifier (`if` or `math`).
        sample_id: Validation sample id.
        sequence_id: Unique sequence identifier (supports multiple rollouts/sample).
        dataset_index: Source dataset row index when available.
        full_ids: Full token ids (`prompt || response`) as 1D tensor.
        prompt_len: Number of prompt tokens inside `full_ids`.
        tokenizer: Tokenizer used for decode diagnostics.
        base_model: Base reference model.
        rl_model: RL fine-tuned model.
        device: Runtime device string.

    Returns:
        List of per-token dictionaries containing log-prob and delta-logp statistics.
    """

    rows: List[Dict[str, Any]] = []

    full_batch = full_ids.unsqueeze(0)
    full_attention = torch.ones_like(full_batch, device=device)

    rl_logits = rl_model(input_ids=full_batch, attention_mask=full_attention).logits
    base_logits = base_model(input_ids=full_batch, attention_mask=full_attention).logits

    # Shift logits by one for autoregressive token likelihood indexing.
    rl_log_probs = torch.log_softmax(rl_logits[:, :-1, :].to(torch.float32), dim=-1)
    base_log_probs = torch.log_softmax(base_logits[:, :-1, :].to(torch.float32), dim=-1)

    full_len = int(full_ids.shape[0])
    positions = torch.arange(prompt_len, full_len, device=device)
    shifted_positions = positions - 1
    target_token_ids = full_ids[positions]

    rl_token_logp = rl_log_probs[0, shifted_positions, target_token_ids]
    base_token_logp = base_log_probs[0, shifted_positions, target_token_ids]
    delta_logp = rl_token_logp - base_token_logp

    for idx in range(int(positions.numel())):
        token_id = int(target_token_ids[idx].item())
        token_text = tokenizer.decode([token_id])
        rows.append(
            {
                "task": task_name,
                "sample_id": int(sample_id),
                "sequence_id": int(sequence_id),
                "dataset_index": int(dataset_index),
                "token_position": int(positions[idx].item()),
                "token_id": token_id,
                "token_text": token_text,
                "logp_rl": float(rl_token_logp[idx].item()),
                "logp_base": float(base_token_logp[idx].item()),
                "delta_logp": float(delta_logp[idx].item()),
            }
        )

    return rows


@torch.no_grad()
def collect_token_delta_records(
    task_name: str,
    validation_df: pd.DataFrame,
    tokenizer: AutoTokenizer,
    base_model: AutoModelForCausalLM,
    rl_model: AutoModelForCausalLM,
    cfg: CriticalTokenConfig,
    device: str,
) -> tuple[pd.DataFrame, dict[int, torch.Tensor], dict[str, Any]]:
    """Collect per-token `Δlog p` records on on-the-fly RL-generated trajectories.

    This is the fallback path when Fisher/persistent rollout artifacts are unavailable.

    Args:
        task_name: Task identifier (`if` or `math`).
        validation_df: Prompt dataframe with `sample_id`, `dataset_index`, `prompt_text`.
        tokenizer: Tokenizer shared by compared checkpoints.
        base_model: Base reference model.
        rl_model: Task RL model.
        cfg: Critical token extraction config.
        device: Runtime device.

    Returns:
        Tuple of:
            - token-level dataframe with `delta_logp`
            - sequence cache mapping `sequence_id -> full_token_ids`
            - summary statistics dictionary
    """

    records: List[Dict[str, Any]] = []
    sequence_cache: dict[int, torch.Tensor] = {}
    generated_token_count = 0

    iterator = tqdm(
        validation_df.itertuples(index=False),
        total=len(validation_df),
        desc=f"Collect Δlogp (online, {task_name})",
    )
    for row in iterator:
        prompt_text = str(row.prompt_text)

        encoded_prompt = tokenizer(
            prompt_text,
            return_tensors="pt",
            truncation=True,
            max_length=cfg.max_prompt_tokens,
        )
        input_ids = encoded_prompt["input_ids"].to(device)
        attention_mask = encoded_prompt["attention_mask"].to(device)

        generated_ids = rl_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

        full_ids = generated_ids[0]
        prompt_len = int(input_ids.shape[1])
        full_len = int(full_ids.shape[0])

        # Skip degenerate cases where generation immediately stops.
        if full_len <= prompt_len:
            continue

        sequence_id = int(row.sample_id)
        sequence_cache[sequence_id] = full_ids.detach().cpu()
        generated_token_count += full_len - prompt_len

        records.extend(
            _compute_token_delta_rows_for_sequence(
                task_name=task_name,
                sample_id=int(row.sample_id),
                sequence_id=sequence_id,
                dataset_index=int(row.dataset_index),
                full_ids=full_ids,
                prompt_len=prompt_len,
                tokenizer=tokenizer,
                base_model=base_model,
                rl_model=rl_model,
                device=device,
            )
        )

    records_df = pd.DataFrame(records)
    summary = {
        "task": task_name,
        "source": "online_generation",
        "num_prompts": int(len(validation_df)),
        "num_sequences": int(len(sequence_cache)),
        "num_token_records": int(len(records_df)),
        "generated_token_count": int(generated_token_count),
    }
    return records_df, sequence_cache, summary


@torch.no_grad()
def collect_token_delta_records_from_fisher_rollouts(
    task_name: str,
    correct_rollout_df: pd.DataFrame,
    tokenizer: AutoTokenizer,
    base_model: AutoModelForCausalLM,
    rl_model: AutoModelForCausalLM,
    cfg: CriticalTokenConfig,
    device: str,
    allowed_sample_ids: set[int] | None,
    sample_id_to_dataset_index: Mapping[int, int] | None,
    use_only_correct_rows: bool,
    max_rollouts_per_sample: int | None,
    max_total_rollouts: int | None,
    sampling_seed: int,
) -> tuple[pd.DataFrame, dict[int, torch.Tensor], dict[str, Any]]:
    """Collect token-level delta-logp records from cached Fisher correct-rollout tables.

    Args:
        task_name: Task identifier (`if` or `math`).
        correct_rollout_df: DataFrame loaded from `correct_rollout_trajectories.parquet`.
        tokenizer: Tokenizer used by both base and RL models.
        base_model: Base model for `log p_base` evaluation.
        rl_model: RL model for `log p_rl` evaluation.
        cfg: Critical token config used for truncation budgets.
        device: Runtime device string.
        allowed_sample_ids: Optional validation sample-id whitelist.
        sample_id_to_dataset_index: Mapping from sample id to dataset index.
        use_only_correct_rows: If `True`, keep only `is_correct=True` when column exists.
        max_rollouts_per_sample: Optional cap to avoid overwhelming repeated rollouts
            for one validation sample.
        max_total_rollouts: Optional global cap on total correct trajectories used
            for this task. This is applied after all filtering steps.
        sampling_seed: Deterministic random seed for rollout-row sampling.

    Returns:
        Tuple of:
            - token-level dataframe
            - sequence cache mapping `sequence_id -> full_token_ids`
            - summary dictionary
    """

    required_cols = {"sample_index", "input", "output"}
    missing_cols = sorted(required_cols - set(correct_rollout_df.columns))
    if missing_cols:
        raise ValueError(
            f"Correct rollout parquet for task '{task_name}' is missing required columns: {missing_cols}"
        )

    working_df = correct_rollout_df.copy()

    # Keep only successful trajectories when metadata is available.
    if use_only_correct_rows and "is_correct" in working_df.columns:
        working_df = working_df[working_df["is_correct"].astype(bool)].copy()

    if allowed_sample_ids is not None:
        working_df = working_df[working_df["sample_index"].isin(sorted(allowed_sample_ids))].copy()

    if max_rollouts_per_sample is not None:
        working_df = (
            working_df.groupby("sample_index", sort=False)
            .head(int(max_rollouts_per_sample))
            .reset_index(drop=True)
        )

    # Apply rollout-level sampling on the global correct-trajectory pool.
    # This is the key behavioral change requested by the user: select N
    # corrected rollout rows directly, instead of selecting prompt rows first.
    if max_total_rollouts is not None and len(working_df) > int(max_total_rollouts):
        working_df = (
            working_df.sample(n=int(max_total_rollouts), random_state=int(sampling_seed), replace=False)
            .sort_values("sample_index")
            .reset_index(drop=True)
        )

    records: List[Dict[str, Any]] = []
    sequence_cache: dict[int, torch.Tensor] = {}

    skipped_empty_output = 0
    skipped_short = 0
    used_rows = 0
    sequence_id_counter = 0

    iterator = tqdm(
        working_df.itertuples(index=False),
        total=len(working_df),
        desc=f"Collect Δlogp (fisher_rollout, {task_name})",
    )

    for row in iterator:
        sample_id = int(getattr(row, "sample_index"))
        prompt_text = str(getattr(row, "input"))
        output_text = str(getattr(row, "output"))

        # Empty response cannot contribute meaningful generated-token statistics.
        if output_text.strip() == "":
            skipped_empty_output += 1
            continue

        encoded_prompt = tokenizer(
            prompt_text,
            return_tensors="pt",
            truncation=True,
            max_length=cfg.max_prompt_tokens,
        )
        prompt_ids = encoded_prompt["input_ids"][0]
        output_ids = tokenizer(
            output_text,
            return_tensors="pt",
            add_special_tokens=False,
            truncation=True,
            max_length=cfg.max_new_tokens,
        )["input_ids"][0]

        # If truncation removes all continuation tokens, skip safely.
        if int(output_ids.numel()) == 0:
            skipped_short += 1
            continue

        full_ids = torch.cat([prompt_ids, output_ids], dim=0).to(device)
        prompt_len = int(prompt_ids.shape[0])
        full_len = int(full_ids.shape[0])
        if full_len <= prompt_len:
            skipped_short += 1
            continue

        sequence_id = int(sequence_id_counter)
        sequence_id_counter += 1

        sequence_cache[sequence_id] = full_ids.detach().cpu()
        dataset_index = (
            int(sample_id_to_dataset_index[sample_id])
            if sample_id_to_dataset_index is not None and sample_id in sample_id_to_dataset_index
            else int(sample_id)
        )

        records.extend(
            _compute_token_delta_rows_for_sequence(
                task_name=task_name,
                sample_id=sample_id,
                sequence_id=sequence_id,
                dataset_index=dataset_index,
                full_ids=full_ids,
                prompt_len=prompt_len,
                tokenizer=tokenizer,
                base_model=base_model,
                rl_model=rl_model,
                device=device,
            )
        )

        used_rows += 1

    records_df = pd.DataFrame(records)
    summary = {
        "task": task_name,
        "source": "fisher_correct_rollout",
        "input_rollout_rows": int(len(correct_rollout_df)),
        "rows_after_filters": int(len(working_df)),
        "used_rollout_rows": int(used_rows),
        "num_sequences": int(len(sequence_cache)),
        "num_token_records": int(len(records_df)),
        "skipped_empty_output": int(skipped_empty_output),
        "skipped_too_short": int(skipped_short),
        "use_only_correct_rows": bool(use_only_correct_rows),
        "max_rollouts_per_sample": max_rollouts_per_sample,
        "max_total_rollouts": max_total_rollouts,
        "sampling_seed": int(sampling_seed),
    }
    return records_df, sequence_cache, summary


def select_critical_tokens(
    token_records_df: pd.DataFrame,
    cfg: CriticalTokenConfig,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Select critical tokens using top-p and epsilon thresholds jointly.

    Args:
        token_records_df: DataFrame with token-level `delta_logp` values.
        cfg: Critical token extraction config.

    Returns:
        Tuple of:
            - critical-token dataframe sorted by descending `delta_logp`
            - selection metadata (thresholds and counts)
    """

    if token_records_df.empty:
        metadata = {
            "quantile_threshold": float("nan"),
            "final_threshold": float("nan"),
            "num_selected": 0,
        }
        return token_records_df.copy(), metadata

    deltas = token_records_df["delta_logp"].to_numpy(dtype=np.float64)

    # Keep positive `Δlog p` mass because JWCM targets RL-amplified tokens.
    quantile_threshold = float(np.quantile(deltas, 1.0 - cfg.top_p))
    final_threshold = float(max(cfg.epsilon, quantile_threshold))

    critical_df = token_records_df[token_records_df["delta_logp"] >= final_threshold].copy()
    critical_df.sort_values("delta_logp", ascending=False, inplace=True)

    # Optional cap keeps exact token-wise attribution tractable.
    if cfg.max_critical_tokens is not None and len(critical_df) > cfg.max_critical_tokens:
        critical_df = critical_df.head(cfg.max_critical_tokens).copy()

    metadata = {
        "quantile_threshold": quantile_threshold,
        "final_threshold": final_threshold,
        "num_selected": int(len(critical_df)),
        "top_p": float(cfg.top_p),
        "epsilon": float(cfg.epsilon),
        "max_critical_tokens": cfg.max_critical_tokens,
    }
    return critical_df, metadata


def save_delta_logp_statistics(
    task_name: str,
    token_records_df: pd.DataFrame,
    critical_df: pd.DataFrame,
    critical_meta: Mapping[str, Any],
    output_dir: Path,
) -> Dict[str, Any]:
    """Save delta-logp analysis artifacts for downstream diagnostics.

    Artifacts include:
    - Task-level summary JSON
    - Quantile table CSV
    - Per-sample distribution CSV
    - Top critical-token table CSV

    Args:
        task_name: Task identifier.
        token_records_df: Full token-delta dataframe for the task.
        critical_df: Critical token dataframe selected by thresholding.
        critical_meta: Selection metadata returned by `select_critical_tokens`.
        output_dir: Destination directory (usually `critical_tokens/`).

    Returns:
        Dictionary containing saved artifact paths and summary statistics.
    """

    output_dir.mkdir(parents=True, exist_ok=True)

    # Ensure numeric stability in summary computations.
    deltas = token_records_df["delta_logp"].to_numpy(dtype=np.float64)
    critical_deltas = critical_df["delta_logp"].to_numpy(dtype=np.float64)

    quantile_levels = [0.0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
    quantile_rows: List[Dict[str, float]] = []
    if deltas.size > 0:
        for q in quantile_levels:
            quantile_rows.append(
                {
                    "quantile": float(q),
                    "delta_logp": float(np.quantile(deltas, q)),
                }
            )

    quantile_df = pd.DataFrame(quantile_rows)
    quantile_csv_path = output_dir / f"{task_name}_delta_logp_quantiles.csv"
    quantile_df.to_csv(quantile_csv_path, index=False)

    per_sample_df = (
        token_records_df.groupby("sample_id", as_index=False)
        .agg(
            token_count=("delta_logp", "size"),
            delta_mean=("delta_logp", "mean"),
            delta_std=("delta_logp", "std"),
            delta_min=("delta_logp", "min"),
            delta_max=("delta_logp", "max"),
            positive_ratio=("delta_logp", lambda s: float((s > 0.0).mean())),
            critical_token_count=(
                "delta_logp",
                lambda s: int((s >= float(critical_meta.get("final_threshold", 0.0))).sum()),
            ),
        )
        .sort_values("delta_max", ascending=False)
        .reset_index(drop=True)
    )
    per_sample_csv_path = output_dir / f"{task_name}_delta_logp_per_sample.csv"
    per_sample_df.to_csv(per_sample_csv_path, index=False)

    top_token_df = (
        critical_df.groupby(["token_id", "token_text"], as_index=False)
        .agg(
            critical_count=("delta_logp", "size"),
            critical_delta_mean=("delta_logp", "mean"),
            critical_delta_max=("delta_logp", "max"),
        )
        .sort_values("critical_count", ascending=False)
        .head(200)
        .reset_index(drop=True)
    )
    top_token_csv_path = output_dir / f"{task_name}_critical_top_tokens.csv"
    top_token_df.to_csv(top_token_csv_path, index=False)

    summary_payload = {
        "task": task_name,
        "num_token_records": int(len(token_records_df)),
        "num_critical_tokens": int(len(critical_df)),
        "critical_ratio": float(len(critical_df) / max(len(token_records_df), 1)),
        "num_unique_samples": int(token_records_df["sample_id"].nunique()),
        "num_unique_sequences": int(token_records_df["sequence_id"].nunique()),
        "positive_delta_ratio": float((deltas > 0.0).mean()) if deltas.size > 0 else 0.0,
        "delta_logp_mean": float(deltas.mean()) if deltas.size > 0 else 0.0,
        "delta_logp_std": float(deltas.std()) if deltas.size > 0 else 0.0,
        "critical_delta_logp_mean": float(critical_deltas.mean()) if critical_deltas.size > 0 else 0.0,
        "critical_delta_logp_std": float(critical_deltas.std()) if critical_deltas.size > 0 else 0.0,
        "critical_selection": dict(critical_meta),
    }
    summary_json_path = output_dir / f"{task_name}_delta_logp_summary.json"
    save_json(summary_payload, summary_json_path)

    return {
        "summary_json": str(summary_json_path),
        "quantiles_csv": str(quantile_csv_path),
        "per_sample_csv": str(per_sample_csv_path),
        "critical_top_tokens_csv": str(top_token_csv_path),
        "summary": summary_payload,
    }


def build_critical_sequence_payload(
    critical_df: pd.DataFrame,
    sequence_cache: Mapping[int, torch.Tensor],
) -> List[Dict[str, Any]]:
    """Group critical token positions by sequence for attribution backprop.

    Args:
        critical_df: Critical token dataframe with `sequence_id` (or `sample_id`) and `token_position`.
        sequence_cache: Mapping from sequence id to generated full token ids.

    Returns:
        List of dictionaries containing sequence tensors and sorted token positions.
    """

    payload: List[Dict[str, Any]] = []
    if critical_df.empty:
        return payload

    grouping_column = "sequence_id" if "sequence_id" in critical_df.columns else "sample_id"

    for sequence_id, group_df in critical_df.groupby(grouping_column):
        if int(sequence_id) not in sequence_cache:
            continue
        positions = sorted(int(value) for value in group_df["token_position"].tolist())
        payload.append(
            {
                "sequence_id": int(sequence_id),
                "sample_id": int(group_df["sample_id"].iloc[0]),
                "full_ids": sequence_cache[int(sequence_id)],
                "critical_positions": positions,
            }
        )

    return payload


In [ ]:
def validate_parameter_compatibility(
    base_model: AutoModelForCausalLM,
    task_model: AutoModelForCausalLM,
) -> None:
    """Validate that named parameter sets and shapes match between two models.

    Args:
        base_model: Base model.
        task_model: Task RL model.

    Returns:
        None. Raises `ValueError` when mismatch is detected.
    """

    base_params = dict(base_model.named_parameters())
    task_params = dict(task_model.named_parameters())

    if set(base_params.keys()) != set(task_params.keys()):
        missing_in_task = sorted(set(base_params.keys()) - set(task_params.keys()))
        missing_in_base = sorted(set(task_params.keys()) - set(base_params.keys()))
        raise ValueError(
            "Named parameter keys mismatch. "
            f"Missing in task: {missing_in_task[:5]}, missing in base: {missing_in_base[:5]}"
        )

    for name in base_params.keys():
        if base_params[name].shape != task_params[name].shape:
            raise ValueError(
                f"Shape mismatch for parameter '{name}': "
                f"base={tuple(base_params[name].shape)} task={tuple(task_params[name].shape)}"
            )


def build_abs_task_vector_cpu(
    base_model: AutoModelForCausalLM,
    task_model: AutoModelForCausalLM,
) -> Dict[str, torch.Tensor]:
    """Build absolute task vector `|Δθ|` on CPU for JWCM attribution.

    Using `|Δθ|` is mathematically equivalent inside `|grad * Δθ|` and avoids
    repeated absolute calls in inner loops.

    Args:
        base_model: Base model used as merge anchor.
        task_model: Task RL model.

    Returns:
        Dictionary mapping parameter names to CPU float32 tensors of `|Δθ|`.
    """

    validate_parameter_compatibility(base_model, task_model)

    abs_delta: Dict[str, torch.Tensor] = {}
    base_params = dict(base_model.named_parameters())
    task_params = dict(task_model.named_parameters())

    for name in tqdm(base_params.keys(), desc="Build |Δθ| (CPU)"):
        base_tensor = base_params[name].detach().to(torch.float32).cpu()
        task_tensor = task_params[name].detach().to(torch.float32).cpu()

        # Keep float32 for stable accumulation in later score updates.
        abs_delta[name] = torch.abs(task_tensor - base_tensor)

    return abs_delta


def initialize_importance_buffers(abs_delta: Mapping[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    """Allocate zero-initialized importance buffers with same shapes as `|Δθ|`.

    Args:
        abs_delta: Parameter-wise absolute task vector dictionary.

    Returns:
        Dictionary of float32 zero tensors for importance accumulation.
    """

    return {name: torch.zeros_like(tensor, dtype=torch.float32) for name, tensor in abs_delta.items()}


def accumulate_importance_from_current_grads(
    named_params: Mapping[str, torch.nn.Parameter],
    abs_delta: Mapping[str, torch.Tensor],
    importance: MutableMapping[str, torch.Tensor],
) -> None:
    """Accumulate `|grad| * |Δθ|` from current model gradients into `importance`.

    Args:
        named_params: Named parameter mapping for the task model.
        abs_delta: Absolute task vector dictionary (`|Δθ|`) on CPU.
        importance: Mutable importance accumulator dictionary on CPU.

    Returns:
        None. Updates `importance` in-place.
    """

    for name, parameter in named_params.items():
        grad = parameter.grad
        if grad is None:
            continue

        # Move gradients to CPU once per step, then accumulate with CPU `|Δθ|`.
        grad_abs_cpu = torch.abs(grad.detach().to(torch.float32).cpu())
        importance[name].add_(grad_abs_cpu * abs_delta[name])


def compute_importance_scores(
    task_model: AutoModelForCausalLM,
    critical_payload: Sequence[Mapping[str, Any]],
    abs_delta: Mapping[str, torch.Tensor],
    cfg: AttributionConfig,
    device: str,
) -> tuple[Dict[str, torch.Tensor], Dict[str, Any]]:
    """Compute JWCM parameter importance `S_j` for one task (GPU-optimized).

    Formula: S_j = Σ_{(x,y_t) in C_τ} |∂log p(y_t|x)/∂θ_j · Δθ_{τ,j}|

    Two modes are supported:
    - `exact_token_abs`: formula-faithful token-wise accumulation.
    - `sequence_sum_approx`: one backward per sequence for lower runtime.

    Performance optimizations (mathematically identical to naive version):
    1. GPU-resident accumulation: `abs_delta` and `importance` stay on GPU,
       eliminating ~310 synchronous CPU↔GPU transfers per backward pass.
       This is the dominant bottleneck in the original implementation because
       each `.cpu()` call is a CUDA synchronization point.
    2. `torch.autograd.grad`: Returns gradient tensors directly without
       populating `.grad` attributes, avoiding `zero_grad()` overhead
       (which iterates all 310 parameters per call).
    3. Selective log_softmax: Only computed at critical token positions
       instead of the full sequence, reducing the retained computation
       graph size and memory for `retain_graph=True` backward passes.
    4. In-place operations: `abs_()` and `mul_()` on intermediate gradient
       tensors minimize transient GPU memory allocations in the inner loop.

    GPU memory budget for Qwen3-1.7B (~310 params, ~1.7B scalars):
    - Model weights (bf16):     ~3.4 GB
    - abs_delta_gpu (fp32):     ~6.8 GB
    - importance_gpu (fp32):    ~6.8 GB
    - Gradients (fp32, temp):   ~6.8 GB (freed after each accumulation)
    - Forward activations:      ~2-4 GB (depends on sequence length)
    - Total peak:               ~26-28 GB (fits on 40GB+ GPUs)

    Args:
        task_model: Task RL model used for gradient computation.
        critical_payload: Grouped critical token data by sequence.
        abs_delta: Absolute task vector dictionary (`|Δθ|`) on CPU float32.
        cfg: Attribution config.
        device: Runtime device.

    Returns:
        Tuple of:
            - importance dictionary (`S_j`) on CPU float32
            - metadata with processed token/sequence counters
    """

    # ── Optimization 1: GPU-resident accumulation buffers ──────────────────────
    # Moving abs_delta and importance to GPU eliminates the per-token
    # `grad.detach().float().cpu()` calls that dominated wall time.
    # Each such call is a CUDA sync point; with ~310 params × N tokens,
    # this created thousands of sync barriers per sequence.
    abs_delta_gpu: Dict[str, torch.Tensor] = {
        name: t.to(device, non_blocking=True)
        for name, t in abs_delta.items()
    }
    importance_gpu: Dict[str, torch.Tensor] = {
        name: torch.zeros_like(t)
        for name, t in abs_delta_gpu.items()
    }

    # ── Optimization 2: Precompute ordered param list for autograd.grad ────────
    # torch.autograd.grad returns gradients directly as a tuple, indexed
    # by the same order as `param_list`. This avoids:
    #   (a) zero_grad() calls (which iterate all params to clear .grad)
    #   (b) .grad attribute lookup overhead per parameter per token
    param_entries = [
        (name, param)
        for name, param in task_model.named_parameters()
        if param.requires_grad
    ]
    param_list = [p for _, p in param_entries]
    param_names = [n for n, _ in param_entries]

    processed_sequences = 0
    processed_tokens = 0
    hit_budget = False

    progress = tqdm(critical_payload, desc=f"Attribution ({cfg.mode})")
    for entry in progress:
        if hit_budget:
            break

        full_ids_cpu = entry["full_ids"]
        critical_positions = [
            int(pos) for pos in entry["critical_positions"] if int(pos) > 0
        ]

        if not critical_positions:
            continue

        # Single unpadded sequence: attention_mask is implicitly all-ones,
        # so we skip creating it to avoid the allocation overhead.
        full_ids = full_ids_cpu.to(device).unsqueeze(0)

        # ── Forward pass (shared across all critical tokens in this sequence) ──
        outputs = task_model(input_ids=full_ids)
        logits = outputs.logits  # (1, seq_len, vocab_size)

        if cfg.mode == "exact_token_abs":
            # ── Optimization 3: Selective log_softmax ──────────────────────
            # Only compute log_softmax at the critical token positions,
            # not across the entire sequence. This shrinks the retained
            # computation graph for retain_graph=True backward passes.
            # Original: log_softmax over (seq_len × vocab) → ~seq_len*vocab floats
            # Optimized: log_softmax over (N_critical × vocab) → ~N_crit*vocab floats
            shifted_positions = [p - 1 for p in critical_positions]
            target_ids = [
                int(full_ids[0, p].item()) for p in critical_positions
            ]

            # Index logits only at critical positions: (N_critical, vocab_size)
            critical_logits = logits[0, shifted_positions, :].to(torch.float32)
            critical_log_probs = torch.log_softmax(critical_logits, dim=-1)

            n_crit = len(critical_positions)
            for token_idx in range(n_crit):
                # Extract scalar log-prob for this critical token.
                scalar_log_prob = critical_log_probs[token_idx, target_ids[token_idx]]

                # Only retain graph if more tokens remain in this sequence.
                retain = token_idx < (n_crit - 1)

                # ── Optimization 2: autograd.grad ──────────────────────────
                # Returns gradient tensors directly without populating .grad,
                # eliminating the zero_grad() → backward() → read .grad cycle.
                # allow_unused=True handles frozen/buffer parameters gracefully.
                grads = torch.autograd.grad(
                    scalar_log_prob,
                    param_list,
                    retain_graph=retain,
                    create_graph=False,
                    allow_unused=True,
                )

                # ── Optimization 1+4: GPU accumulation with in-place ops ──
                # S_j += |∂log p / ∂θ_j| * |Δθ_j|
                # g.float() creates fp32 copy; .abs_() and .mul_() are in-place
                # on that copy, so only one transient tensor per parameter.
                for i, g in enumerate(grads):
                    if g is not None:
                        name = param_names[i]
                        if name in importance_gpu:
                            g_abs = g.to(torch.float32).abs_()
                            importance_gpu[name].add_(g_abs.mul_(abs_delta_gpu[name]))

                processed_tokens += 1
                if cfg.max_backprop_tokens is not None and processed_tokens >= cfg.max_backprop_tokens:
                    hit_budget = True
                    break

        elif cfg.mode == "sequence_sum_approx":
            # Approximation: sum critical log-probs and perform one backward.
            # This computes S_j ≈ |Σ_t grad_j^(t)| * |Δθ_j| instead of
            # the exact Σ_t |grad_j^(t)| * |Δθ_j|. The triangle inequality
            # means this is a lower bound on the exact score.
            log_probs = torch.log_softmax(
                logits[:, :-1, :].to(torch.float32), dim=-1
            )

            scalar_terms: List[torch.Tensor] = []
            for token_position in critical_positions:
                shifted = token_position - 1
                target_id = int(full_ids[0, token_position].item())
                scalar_terms.append(log_probs[0, shifted, target_id])

            summed_loss = torch.stack(scalar_terms).sum()

            # Single backward pass for the entire sequence.
            grads = torch.autograd.grad(
                summed_loss,
                param_list,
                retain_graph=False,
                create_graph=False,
                allow_unused=True,
            )

            # GPU-side accumulation (same as exact mode).
            for i, g in enumerate(grads):
                if g is not None:
                    name = param_names[i]
                    if name in importance_gpu:
                        g_abs = g.to(torch.float32).abs_()
                        importance_gpu[name].add_(g_abs.mul_(abs_delta_gpu[name]))

            processed_tokens += len(critical_positions)
        else:
            raise ValueError(f"Unknown attribution mode: {cfg.mode}")

        processed_sequences += 1

        # Free forward outputs to reduce memory between sequences.
        del outputs, logits

        # Periodic CUDA cache cleanup to prevent fragmentation.
        if processed_sequences % 64 == 0:
            gc.collect()
            torch.cuda.empty_cache()

        if cfg.max_backprop_tokens is not None and processed_tokens >= cfg.max_backprop_tokens:
            hit_budget = True

    # ── Transfer final importance scores to CPU (once, not per-token) ──────────
    importance: Dict[str, torch.Tensor] = {
        name: t.cpu() for name, t in importance_gpu.items()
    }

    if cfg.normalize_by_token_count and processed_tokens > 0:
        scale = float(processed_tokens)
        for name in importance.keys():
            importance[name].div_(scale)

    # ── Free GPU accumulation buffers ──────────────────────────────────────────
    del abs_delta_gpu, importance_gpu
    gc.collect()
    torch.cuda.empty_cache()

    metadata = {
        "mode": cfg.mode,
        "processed_sequences": int(processed_sequences),
        "processed_tokens": int(processed_tokens),
        "max_backprop_tokens": cfg.max_backprop_tokens,
        "normalize_by_token_count": bool(cfg.normalize_by_token_count),
    }
    return importance, metadata


In [ ]:
def estimate_sparsify_kappa(
    importance_by_task: Mapping[str, Mapping[str, torch.Tensor]],
    quantile: float,
    sampled_values_per_tensor: int,
    seed: int,
) -> float:
    """Estimate global sparsification threshold `kappa` from sampled importance values.

    Args:
        importance_by_task: Nested mapping `task -> parameter -> importance tensor`.
        quantile: Target quantile in `[0, 1]`.
        sampled_values_per_tensor: Maximum sampled entries per parameter tensor.
        seed: Random seed for deterministic sampling.

    Returns:
        Scalar `kappa` value used for element-wise sparsification mask.
    """

    if not importance_by_task:
        return 0.0

    rng = np.random.default_rng(seed)
    task_names = list(importance_by_task.keys())
    param_names = list(next(iter(importance_by_task.values())).keys())

    sampled_chunks: List[np.ndarray] = []
    for param_name in tqdm(param_names, desc="Sample kappa statistics"):
        max_tensor = None
        for task_name in task_names:
            task_tensor = importance_by_task[task_name][param_name]
            max_tensor = task_tensor if max_tensor is None else torch.maximum(max_tensor, task_tensor)

        flat = max_tensor.reshape(-1).cpu().numpy()
        if flat.size == 0:
            continue

        if flat.size > sampled_values_per_tensor:
            indices = rng.choice(flat.size, size=sampled_values_per_tensor, replace=False)
            sampled = flat[indices]
        else:
            sampled = flat

        sampled_chunks.append(sampled.astype(np.float64, copy=False))

    if not sampled_chunks:
        return 0.0

    concatenated = np.concatenate(sampled_chunks, axis=0)
    return float(np.quantile(concatenated, quantile))


def merge_with_soft_attribution_inplace(
    base_model: AutoModelForCausalLM,
    task_models: Mapping[str, AutoModelForCausalLM],
    importance_by_task: Mapping[str, Mapping[str, torch.Tensor]],
    epsilon: float,
    sparsify_kappa: float | None = None,
) -> Dict[str, Any]:
    """Apply JWCM v2 soft attribution merge in-place.

    Formula per parameter coordinate:
    `θ_base + Σ(S * Δθ) / (ΣS + epsilon)`

    Optional sparsification mask:
    `1[max_task S > kappa]`

    Args:
        base_model: Base model to overwrite with merged parameters.
        task_models: Mapping from task name to tuned checkpoint model.
        importance_by_task: Mapping `task -> parameter -> S tensor`.
        epsilon: Numerical stabilizer for denominator.
        sparsify_kappa: Optional threshold for sparsification. If `None`, no mask.

    Returns:
        Dictionary with merge summary statistics.
    """

    base_named = dict(base_model.named_parameters())
    task_named = {task_name: dict(model.named_parameters()) for task_name, model in task_models.items()}
    task_names = list(task_models.keys())

    num_mask_active = 0
    num_total = 0

    with torch.no_grad():
        for name, base_param in tqdm(base_named.items(), desc="Merge JWCM v2"):
            if not torch.is_floating_point(base_param.data):
                continue

            base_fp32 = base_param.data.detach().to(torch.float32)
            numerator = torch.zeros_like(base_fp32)
            denominator = torch.zeros_like(base_fp32)
            max_importance = torch.zeros_like(base_fp32)

            for task_name in task_names:
                task_param = task_named[task_name][name].data.detach().to(torch.float32)
                delta = task_param - base_fp32
                importance = importance_by_task[task_name][name].to(torch.float32)

                numerator.add_(importance * delta)
                denominator.add_(importance)
                max_importance = torch.maximum(max_importance, importance)

            merged_delta = numerator / (denominator + epsilon)

            if sparsify_kappa is not None:
                mask = (max_importance > float(sparsify_kappa)).to(torch.float32)
                merged_delta = merged_delta * mask

                num_mask_active += int(mask.sum().item())
                num_total += int(mask.numel())

            merged_tensor = base_fp32 + merged_delta
            base_param.data.copy_(merged_tensor.to(base_param.dtype))

    summary = {
        "sparsify_enabled": sparsify_kappa is not None,
        "sparsify_kappa": None if sparsify_kappa is None else float(sparsify_kappa),
        "mask_active_ratio": None if num_total == 0 else float(num_mask_active / num_total),
    }
    return summary


def save_merged_artifacts(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    output_dir: Path,
    metadata: Mapping[str, Any],
) -> None:
    """Save merged model, tokenizer, and merge metadata.

    Args:
        model: Merged model object.
        tokenizer: Tokenizer to save alongside checkpoint.
        output_dir: Destination directory.
        metadata: JSON metadata payload.

    Returns:
        None. Artifacts are persisted to disk.
    """

    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir, safe_serialization=True)
    tokenizer.save_pretrained(output_dir)
    save_json(dict(metadata), output_dir / "merge_metadata.json")


In [ ]:
set_seed(RUNTIME.seed)

a_dirs = {
    "validation": RUNTIME.output_root / "validation",
    "critical": RUNTIME.output_root / "critical_tokens",
    "importance": RUNTIME.output_root / "importance",
    "metadata": RUNTIME.output_root / "metadata",
}
for path in a_dirs.values():
    path.mkdir(parents=True, exist_ok=True)

validation_tables: Dict[str, pd.DataFrame] = {}
validation_paths: Dict[str, str] = {}
validation_sample_id_sets: Dict[str, set[int]] = {}

for task_spec in TASK_SPECS:
    source_validation_path = task_spec.fisher_validation_path
    source_validation_df = pd.read_parquet(source_validation_path)

    required_columns = {
        "sample_id",
        "task",
        "dataset_index",
        "prompt_messages_json",
        "prompt_text",
    }
    missing_columns = sorted(required_columns - set(source_validation_df.columns))
    if missing_columns:
        raise ValueError(
            f"Fisher validation parquet for task '{task_spec.name}' is missing required columns: "
            f"{missing_columns}"
        )

    # Copy Fisher validation parquet into this run's output tree for immutable provenance.
    validation_path = a_dirs["validation"] / f"{task_spec.name}_validation.parquet"
    source_validation_df.to_parquet(validation_path, index=False)

    if ATTRIBUTION_VALIDATION_SAMPLES_PER_TASK is None:
        selected_validation_df = source_validation_df.copy()
    else:
        selected_validation_df = source_validation_df.head(ATTRIBUTION_VALIDATION_SAMPLES_PER_TASK).copy()

    if selected_validation_df.empty:
        raise ValueError(
            f"Validation split for task '{task_spec.name}' is empty after selection. "
            f"source_rows={len(source_validation_df)}, selected_n={ATTRIBUTION_VALIDATION_SAMPLES_PER_TASK}"
        )

    selected_validation_df["sample_id"] = selected_validation_df["sample_id"].astype(np.int64)
    selected_validation_df["dataset_index"] = selected_validation_df["dataset_index"].astype(np.int64)

    validation_tables[task_spec.name] = selected_validation_df
    validation_paths[task_spec.name] = str(validation_path)
    validation_sample_id_sets[task_spec.name] = set(
        int(value) for value in selected_validation_df["sample_id"].tolist()
    )

    print(
        f"Task={task_spec.name} validation reuse: "
        f"source_rows={len(source_validation_df)}, "
        f"selected_rows={len(selected_validation_df)}, "
        f"saved_path={validation_path}"
    )


In [ ]:
# Load base model once for all task-level critical token scans.
base_model, _ = load_causal_lm(
    model_name_or_path=RUNTIME.base_model_id,
    torch_dtype=RUNTIME.model_dtype,
    device=RUNTIME.device,
)

importance_paths: Dict[str, str] = {}
run_summary: Dict[str, Any] = {
    "created_at": now_iso(),
    "runtime": {
        "base_model_id": RUNTIME.base_model_id,
        "output_root": str(RUNTIME.output_root),
        "device": RUNTIME.device,
        "dtype": str(RUNTIME.model_dtype),
        "seed": RUNTIME.seed,
    },
    "critical_config": asdict(CRITICAL_CFG),
    "attribution_config": asdict(ATTR_CFG),
    "merge_config": asdict(MERGE_CFG),
    "token_delta_source_config": asdict(TOKEN_SOURCE_CFG),
    "attribution_correct_rollout_samples_per_task": ATTRIBUTION_CORRECT_ROLLOUT_SAMPLES_PER_TASK,
    "fisher_validation_source_paths": {
        task_spec.name: str(task_spec.fisher_validation_path)
        for task_spec in TASK_SPECS
    },
    "fisher_correct_rollout_paths": {
        task_spec.name: None
        if task_spec.fisher_correct_rollout_path is None
        else str(task_spec.fisher_correct_rollout_path)
        for task_spec in TASK_SPECS
    },
    "legacy_precomputed_token_root": str(LEGACY_PRECOMPUTED_TOKEN_ROOT),
    "validation_paths": validation_paths,
    "task_summaries": {},
}


In [ ]:
for task_spec in TASK_SPECS:
    print(f"\n=== Task: {task_spec.name} ===")

    selected_sample_ids = validation_sample_id_sets[task_spec.name]
    selected_validation_df = validation_tables[task_spec.name]

    sample_id_to_dataset_index = {
        int(row.sample_id): int(row.dataset_index)
        for row in selected_validation_df.itertuples(index=False)
    }

    # Load one task model at a time to control peak memory.
    task_model, task_tokenizer = load_causal_lm(
        model_name_or_path=task_spec.model_path,
        torch_dtype=RUNTIME.model_dtype,
        device=RUNTIME.device,
    )

    token_records_df = pd.DataFrame()
    sequence_cache: Dict[int, torch.Tensor] = {}
    token_summary: Dict[str, Any] = {}
    token_delta_source = "unresolved"

    # ------------------------------------------------------------------
    # Source 1 (preferred): Fisher-pipeline correct rollout trajectories
    # ------------------------------------------------------------------
    rollout_path = task_spec.fisher_correct_rollout_path
    if TOKEN_SOURCE_CFG.prefer_fisher_correct_rollout and rollout_path is not None and rollout_path.exists():
        print(f"Loading Fisher correct rollouts: {rollout_path}")
        correct_rollout_df = pd.read_parquet(rollout_path)
        print(
            f"Task={task_spec.name} | source_rows={len(selected_validation_df)} | "
            f"correct_rollout_rows={len(correct_rollout_df)}"
        )
        token_records_df, sequence_cache, token_summary = collect_token_delta_records_from_fisher_rollouts(
            task_name=task_spec.name,
            correct_rollout_df=correct_rollout_df,
            tokenizer=task_tokenizer,
            base_model=base_model,
            rl_model=task_model,
            cfg=CRITICAL_CFG,
            device=RUNTIME.device,
            allowed_sample_ids=selected_sample_ids,
            sample_id_to_dataset_index=sample_id_to_dataset_index,
            use_only_correct_rows=TOKEN_SOURCE_CFG.use_only_correct_rows,
            max_rollouts_per_sample=TOKEN_SOURCE_CFG.max_rollouts_per_sample,
            max_total_rollouts=ATTRIBUTION_CORRECT_ROLLOUT_SAMPLES_PER_TASK,
            sampling_seed=RUNTIME.seed,
        )
        if not token_records_df.empty:
            token_delta_source = "fisher_correct_rollout"
            token_summary["rollout_path"] = str(rollout_path)
            print(
                "Loaded token deltas from Fisher correct rollouts: "
                f"task={task_spec.name}, token_rows={len(token_records_df)}, "
                f"sequence_count={len(sequence_cache)}, selected_samples={len(selected_sample_ids)}, "
                f"sampled_rollouts={token_summary.get('rows_after_filters', -1)}"
            )

    # ------------------------------------------------------------------
    # Source 2 (fallback): previously precomputed token-delta artifacts
    # ------------------------------------------------------------------
    if token_records_df.empty and TOKEN_SOURCE_CFG.fallback_to_precomputed:
        precompute_task_dir = LEGACY_PRECOMPUTED_TOKEN_ROOT / task_spec.name
        precomputed_token_csv = precompute_task_dir / "all_token_deltas.csv"
        precomputed_sequence_cache = precompute_task_dir / "sequence_cache.pt"
        precomputed_summary_json = precompute_task_dir / "token_summary.json"

        required_files = [
            precomputed_token_csv,
            precomputed_sequence_cache,
            precomputed_summary_json,
        ]
        if all(path.exists() for path in required_files):
            token_records_df = pd.read_csv(precomputed_token_csv)
            raw_sequence_cache = torch.load(precomputed_sequence_cache, map_location="cpu")
            sequence_cache = {
                int(sample_id): full_ids
                for sample_id, full_ids in raw_sequence_cache.items()
            }
            token_summary = load_json(precomputed_summary_json)

            # Ensure schema compatibility with multi-sequence payload format.
            if "sequence_id" not in token_records_df.columns:
                token_records_df["sequence_id"] = token_records_df["sample_id"].astype(np.int64)

            token_records_df = token_records_df[
                token_records_df["sample_id"].isin(selected_sample_ids)
            ].copy()

            valid_sequence_ids = set(int(v) for v in token_records_df["sequence_id"].unique().tolist())
            sequence_cache = {
                int(sequence_id): full_ids
                for sequence_id, full_ids in sequence_cache.items()
                if int(sequence_id) in valid_sequence_ids
            }

            if not token_records_df.empty:
                token_delta_source = "legacy_precomputed"
                token_summary["precomputed_token_dir"] = str(precompute_task_dir)
                print(
                    "Loaded token deltas from legacy precompute fallback: "
                    f"task={task_spec.name}, token_rows={len(token_records_df)}, "
                    f"sequence_count={len(sequence_cache)}, selected_samples={len(selected_sample_ids)}"
                )
        else:
            missing_files = [str(path) for path in required_files if not path.exists()]
            print(
                f"Legacy precomputed fallback unavailable for task '{task_spec.name}'; missing files: "
                f"{missing_files}"
            )

    # ------------------------------------------------------------------
    # Source 3 (last resort): online generation on selected validation rows
    # ------------------------------------------------------------------
    if token_records_df.empty and TOKEN_SOURCE_CFG.fallback_to_online_generation:
        print(
            f"Falling back to online generation for task '{task_spec.name}' because "
            "no reusable token-delta source was available."
        )
        token_records_df, sequence_cache, token_summary = collect_token_delta_records(
            task_name=task_spec.name,
            validation_df=selected_validation_df,
            tokenizer=task_tokenizer,
            base_model=base_model,
            rl_model=task_model,
            cfg=CRITICAL_CFG,
            device=RUNTIME.device,
        )
        token_delta_source = "online_generation"

    if token_records_df.empty:
        raise ValueError(
            f"Token-delta records are empty for task '{task_spec.name}' after all source fallbacks."
        )

    # Normalize data types to avoid downstream merge/groupby ambiguity.
    token_records_df["sample_id"] = token_records_df["sample_id"].astype(np.int64)
    token_records_df["sequence_id"] = token_records_df["sequence_id"].astype(np.int64)
    token_records_df["token_position"] = token_records_df["token_position"].astype(np.int64)
    token_records_df["token_id"] = token_records_df["token_id"].astype(np.int64)

    critical_df, critical_meta = select_critical_tokens(
        token_records_df=token_records_df,
        cfg=CRITICAL_CFG,
    )

    critical_payload = build_critical_sequence_payload(
        critical_df=critical_df,
        sequence_cache=sequence_cache,
    )

    if not critical_payload:
        raise ValueError(
            f"No critical sequence payload was built for task '{task_spec.name}'. "
            "Check token alignment and threshold settings."
        )

    critical_csv_path = a_dirs["critical"] / f"{task_spec.name}_critical_tokens.csv"
    token_csv_path = a_dirs["critical"] / f"{task_spec.name}_all_token_deltas.csv"
    critical_df.to_csv(critical_csv_path, index=False)
    token_records_df.to_csv(token_csv_path, index=False)

    delta_logp_artifacts = save_delta_logp_statistics(
        task_name=task_spec.name,
        token_records_df=token_records_df,
        critical_df=critical_df,
        critical_meta=critical_meta,
        output_dir=a_dirs["critical"],
    )

    abs_delta = build_abs_task_vector_cpu(
        base_model=base_model,
        task_model=task_model,
    )

    importance, importance_meta = compute_importance_scores(
        task_model=task_model,
        critical_payload=critical_payload,
        abs_delta=abs_delta,
        cfg=ATTR_CFG,
        device=RUNTIME.device,
    )

    importance_path = a_dirs["importance"] / f"importance_{task_spec.name}.pt"
    torch.save(importance, importance_path)
    importance_paths[task_spec.name] = str(importance_path)

    run_summary["task_summaries"][task_spec.name] = {
        "task_model_path": str(task_spec.model_path),
        "token_delta_source": token_delta_source,
        "token_summary": token_summary,
        "critical_selection": critical_meta,
        "critical_token_csv": str(critical_csv_path),
        "token_delta_csv": str(token_csv_path),
        "token_delta_stats": delta_logp_artifacts,
        "importance_path": str(importance_path),
        "importance_meta": importance_meta,
        "num_selected_validation_samples": int(len(selected_sample_ids)),
        "num_unique_token_delta_samples": int(token_records_df["sample_id"].nunique()),
        "num_unique_token_delta_sequences": int(token_records_df["sequence_id"].nunique()),
    }

    # Aggressive cleanup is important for long notebook sessions.
    del importance
    del abs_delta
    del critical_payload
    del sequence_cache
    del task_model
    del task_tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# Base model no longer needed on device after importance computation.
del base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load importance artifacts for merge stage.
importance_by_task = {task_spec.name: torch.load(importance_paths[task_spec.name], map_location="cpu") for task_spec in TASK_SPECS}

# Load models on CPU for deterministic merge and save.
merge_base_model, merge_tokenizer = load_causal_lm(
    model_name_or_path=RUNTIME.base_model_id,
    torch_dtype=RUNTIME.model_dtype,
    device="cpu",
)
merge_task_models = {}
for task_spec in TASK_SPECS:
    model, _ = load_causal_lm(
        model_name_or_path=task_spec.model_path,
        torch_dtype=RUNTIME.model_dtype,
        device="cpu",
    )
    merge_task_models[task_spec.name] = model

# 1) JWCM v2 soft merge (no sparsification)
soft_summary = merge_with_soft_attribution_inplace(
    base_model=merge_base_model,
    task_models=merge_task_models,
    importance_by_task=importance_by_task,
    epsilon=MERGE_CFG.epsilon,
    sparsify_kappa=None,
)

soft_output_dir = RUNTIME.output_root / "jwcm_v2_soft"
save_merged_artifacts(
    model=merge_base_model,
    tokenizer=merge_tokenizer,
    output_dir=soft_output_dir,
    metadata={
        "created_at": now_iso(),
        "method": "jwcm_v2_soft",
        "runtime": {
            "base_model_id": RUNTIME.base_model_id,
            "output_root": str(RUNTIME.output_root),
            "seed": RUNTIME.seed,
            "model_dtype": str(RUNTIME.model_dtype),
            "device": RUNTIME.device,
        },
        "critical_config": asdict(CRITICAL_CFG),
        "attribution_config": asdict(ATTR_CFG),
        "merge_config": asdict(MERGE_CFG),
        "token_delta_source_config": asdict(TOKEN_SOURCE_CFG),
        "task_models": {task_spec.name: str(task_spec.model_path) for task_spec in TASK_SPECS},
        "importance_paths": importance_paths,
        "merge_summary": soft_summary,
    },
)
print(f"Saved soft merge checkpoint: {soft_output_dir}")

# Re-load base model to ensure sparsified merge starts from the same anchor.
del merge_base_model
gc.collect()

merge_base_model, merge_tokenizer = load_causal_lm(
    model_name_or_path=RUNTIME.base_model_id,
    torch_dtype=RUNTIME.model_dtype,
    device="cpu",
)

if MERGE_CFG.sparsify_mode == "absolute":
    sparsify_kappa = float(MERGE_CFG.sparsify_kappa)
elif MERGE_CFG.sparsify_mode == "sampled_quantile":
    sparsify_kappa = estimate_sparsify_kappa(
        importance_by_task=importance_by_task,
        quantile=MERGE_CFG.sparsify_quantile,
        sampled_values_per_tensor=MERGE_CFG.sampled_values_per_tensor,
        seed=RUNTIME.seed,
    )
else:
    raise ValueError(f"Unsupported sparsify mode: {MERGE_CFG.sparsify_mode}")

sparsified_summary = merge_with_soft_attribution_inplace(
    base_model=merge_base_model,
    task_models=merge_task_models,
    importance_by_task=importance_by_task,
    epsilon=MERGE_CFG.epsilon,
    sparsify_kappa=sparsify_kappa,
)

sparse_output_dir = RUNTIME.output_root / "jwcm_v2_soft_sparsified"
save_merged_artifacts(
    model=merge_base_model,
    tokenizer=merge_tokenizer,
    output_dir=sparse_output_dir,
    metadata={
        "created_at": now_iso(),
        "method": "jwcm_v2_soft_sparsified",
        "runtime": {
            "base_model_id": RUNTIME.base_model_id,
            "output_root": str(RUNTIME.output_root),
            "seed": RUNTIME.seed,
            "model_dtype": str(RUNTIME.model_dtype),
            "device": RUNTIME.device,
        },
        "critical_config": asdict(CRITICAL_CFG),
        "attribution_config": asdict(ATTR_CFG),
        "merge_config": asdict(MERGE_CFG),
        "token_delta_source_config": asdict(TOKEN_SOURCE_CFG),
        "task_models": {task_spec.name: str(task_spec.model_path) for task_spec in TASK_SPECS},
        "importance_paths": importance_paths,
        "sparsify_kappa": float(sparsify_kappa),
        "merge_summary": sparsified_summary,
    },
)
print(f"Saved sparsified merge checkpoint: {sparse_output_dir}")

run_summary.update(
    {
        "importance_paths": importance_paths,
        "soft_output_dir": str(soft_output_dir),
        "sparse_output_dir": str(sparse_output_dir),
        "sparsify_kappa": float(sparsify_kappa),
        "token_delta_source_by_task": {
            task_name: task_summary.get("token_delta_source")
            for task_name, task_summary in run_summary["task_summaries"].items()
        },
        "completed_at": now_iso(),
    }
)
save_json(run_summary, a_dirs["metadata"] / "jwcm_v2_run_summary.json")
print(f"Saved run summary: {a_dirs['metadata'] / 'jwcm_v2_run_summary.json'}")

# Final cleanup.
del merge_base_model
del merge_tokenizer
for _task_name in list(merge_task_models.keys()):
    del merge_task_models[_task_name]
del importance_by_task
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("JWCM v2 pipeline finished.")


## Importance Sparsity and Overlap Analysis

This section visualizes how sparse the saved JWCM importance tensors are and how much high-importance support overlaps between `if` and `math` tasks.


In [ ]:
import re
import matplotlib.pyplot as plt


def infer_layer_bucket(parameter_name: str) -> str:
    """Map a parameter name into a readable layer bucket.

    Args:
        parameter_name: Full model parameter name from `named_parameters()`.

    Returns:
        Bucket label. Decoder blocks use `layer_<idx>`, while non-block tensors
        are grouped into coarse labels for simpler plots.
    """

    # Decoder block tensors are indexed explicitly as `layers.<idx>`.
    layer_match = re.search(r"layers\.(\d+)\.", parameter_name)
    if layer_match is not None:
        layer_index = int(layer_match.group(1))
        return f"layer_{layer_index:02d}"

    # Keep non-decoder tensors visible in separate coarse groups.
    if "embed_tokens" in parameter_name:
        return "embeddings"
    if "lm_head" in parameter_name:
        return "lm_head"
    if "norm" in parameter_name:
        return "final_norm"
    return "other"


def parse_layer_index(layer_bucket: str) -> int | None:
    """Parse decoder layer index from a bucket label.

    Args:
        layer_bucket: Bucket label produced by `infer_layer_bucket`.

    Returns:
        Integer layer index for decoder buckets, otherwise `None`.
    """

    layer_match = re.match(r"layer_(\d+)$", layer_bucket)
    if layer_match is None:
        return None
    return int(layer_match.group(1))


def _sorted_layer_order(labels: Sequence[str]) -> List[str]:
    """Sort bucket labels with decoder layers first in numeric order.

    Args:
        labels: Unordered bucket labels.

    Returns:
        Deterministically sorted labels.
    """

    def _key(label: str) -> tuple[int, int | str]:
        layer_index = parse_layer_index(label)
        if layer_index is not None:
            return (0, layer_index)
        return (1, label)

    return sorted(set(labels), key=_key)


def estimate_task_threshold(
    importance_tensors: Mapping[str, torch.Tensor],
    quantile: float,
    sampled_values_per_tensor: int,
    seed: int,
) -> float:
    """Estimate task-wise high-importance threshold by sampled quantile.

    Args:
        importance_tensors: Mapping `parameter_name -> importance tensor`.
        quantile: Target quantile in `[0, 1]`.
        sampled_values_per_tensor: Max sampled coordinates per tensor.
        seed: RNG seed for reproducibility.

    Returns:
        Scalar threshold used to define "high-importance" coordinates.
    """

    rng = np.random.default_rng(seed)
    sampled_chunks: List[np.ndarray] = []

    # Sample each tensor to avoid multi-billion-element concatenation in RAM.
    for tensor in importance_tensors.values():
        flat = tensor.detach().reshape(-1).cpu().numpy()
        if flat.size == 0:
            continue

        if flat.size > sampled_values_per_tensor:
            sample_indices = rng.choice(flat.size, size=sampled_values_per_tensor, replace=False)
            sampled = flat[sample_indices]
        else:
            sampled = flat

        sampled_chunks.append(sampled.astype(np.float64, copy=False))

    if not sampled_chunks:
        return 0.0

    pooled = np.concatenate(sampled_chunks, axis=0)
    return float(np.quantile(pooled, quantile))


# Load saved importance tensors if they are not already in memory.
if "importance_by_task" not in globals() or not importance_by_task:
    if "importance_paths" not in globals() or not importance_paths:
        raise RuntimeError(
            "importance_by_task and importance_paths are both unavailable. "
            "Run attribution first to generate importance tensors."
        )
    importance_by_task = {
        task_name: torch.load(path, map_location="cpu")
        for task_name, path in importance_paths.items()
    }

analysis_dir = RUNTIME.output_root / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

task_names = sorted(list(importance_by_task.keys()))
if set(task_names) != {"if", "math"}:
    raise ValueError(f"Expected exactly tasks {{'if', 'math'}}, but got {task_names}")

param_names = list(importance_by_task[task_names[0]].keys())

# -------------------------------------------------------------------------
# 1) Per-parameter sparsity/scale summary
# -------------------------------------------------------------------------
sparsity_threshold = 0.0
parameter_rows: List[Dict[str, Any]] = []

for parameter_name in tqdm(param_names, desc="Build per-parameter importance statistics"):
    reference_tensor = importance_by_task[task_names[0]][parameter_name]
    numel = int(reference_tensor.numel())

    row: Dict[str, Any] = {
        "parameter": parameter_name,
        "layer_bucket": infer_layer_bucket(parameter_name),
        "numel": numel,
    }

    for task_name in task_names:
        tensor = importance_by_task[task_name][parameter_name].to(torch.float32)
        nonzero_count = int((tensor > sparsity_threshold).sum().item())

        row[f"{task_name}_nonzero_count"] = nonzero_count
        row[f"{task_name}_nonzero_ratio"] = nonzero_count / max(numel, 1)
        row[f"{task_name}_mean"] = float(tensor.mean().item())
        row[f"{task_name}_max"] = float(tensor.max().item())

    parameter_rows.append(row)

parameter_stats_df = pd.DataFrame(parameter_rows)
parameter_stats_csv = analysis_dir / "importance_parameter_stats.csv"
parameter_stats_df.to_csv(parameter_stats_csv, index=False)
print(f"Saved per-parameter statistics: {parameter_stats_csv}")

# -------------------------------------------------------------------------
# 2) High-importance overlap analysis between IF and Math tasks
# -------------------------------------------------------------------------
high_importance_quantile = 0.995
sampled_values_per_tensor = 8192

thresholds = {
    task_name: estimate_task_threshold(
        importance_tensors=importance_by_task[task_name],
        quantile=high_importance_quantile,
        sampled_values_per_tensor=sampled_values_per_tensor,
        seed=RUNTIME.seed,
    )
    for task_name in task_names
}

print(
    "High-importance thresholds: "
    + ", ".join([f"{task_name}={thresholds[task_name]:.3e}" for task_name in task_names])
)

overlap_rows: List[Dict[str, Any]] = []
global_counts = {
    "if_only": 0,
    "math_only": 0,
    "both": 0,
    "neither": 0,
}

for parameter_name in tqdm(param_names, desc="Compute high-importance overlap"):
    if_tensor = importance_by_task["if"][parameter_name].to(torch.float32)
    math_tensor = importance_by_task["math"][parameter_name].to(torch.float32)

    if_high = if_tensor >= thresholds["if"]
    math_high = math_tensor >= thresholds["math"]

    both_high = if_high & math_high
    if_only = if_high & (~math_high)
    math_only = math_high & (~if_high)
    union_high = if_high | math_high

    both_count = int(both_high.sum().item())
    if_only_count = int(if_only.sum().item())
    math_only_count = int(math_only.sum().item())
    union_count = int(union_high.sum().item())
    numel = int(if_tensor.numel())

    global_counts["both"] += both_count
    global_counts["if_only"] += if_only_count
    global_counts["math_only"] += math_only_count
    global_counts["neither"] += int(numel - union_count)

    overlap_rows.append(
        {
            "parameter": parameter_name,
            "layer_bucket": infer_layer_bucket(parameter_name),
            "numel": numel,
            "if_high_count": int(if_high.sum().item()),
            "math_high_count": int(math_high.sum().item()),
            "if_only_count": if_only_count,
            "math_only_count": math_only_count,
            "both_high_count": both_count,
            "union_high_count": union_count,
            "if_high_ratio": float(if_high.float().mean().item()),
            "math_high_ratio": float(math_high.float().mean().item()),
            "if_only_ratio": if_only_count / max(numel, 1),
            "math_only_ratio": math_only_count / max(numel, 1),
            "both_high_ratio": both_count / max(numel, 1),
            "jaccard": both_count / max(union_count, 1),
        }
    )

overlap_df = pd.DataFrame(overlap_rows)
overlap_csv = analysis_dir / "importance_overlap_stats.csv"
overlap_df.to_csv(overlap_csv, index=False)
print(f"Saved overlap statistics: {overlap_csv}")

# Global IF/Math overlap composition remains useful as a scalar summary.
global_df = pd.DataFrame(
    [
        {"bucket": "if_only", "count": global_counts["if_only"]},
        {"bucket": "math_only", "count": global_counts["math_only"]},
        {"bucket": "both", "count": global_counts["both"]},
        {"bucket": "neither", "count": global_counts["neither"]},
    ]
)
global_df["ratio"] = global_df["count"] / max(global_df["count"].sum(), 1)

fig, ax = plt.subplots(figsize=(8.0, 4.0))
ax.bar(global_df["bucket"], global_df["ratio"], color=["#1f77b4", "#ff7f0e", "#2ca02c", "#7f7f7f"])
ax.set_ylim(0.0, 1.0)
ax.set_ylabel("Coordinate ratio")
ax.set_title("High-Importance Coordinate Overlap (IF vs Math)")
for idx, ratio in enumerate(global_df["ratio"].tolist()):
    ax.text(idx, ratio + 0.01, f"{ratio:.2%}", ha="center", va="bottom", fontsize=9)
fig.tight_layout()
global_overlap_png = analysis_dir / "importance_global_overlap_ratio.png"
fig.savefig(global_overlap_png, dpi=220)
plt.show()
print(f"Saved global overlap plot: {global_overlap_png}")

# -------------------------------------------------------------------------
# 3) Layer-axis histograms for high-importance locations
# -------------------------------------------------------------------------
layer_overlap_df = (
    overlap_df.groupby("layer_bucket", as_index=False)
    .agg(
        numel=("numel", "sum"),
        if_high_count=("if_high_count", "sum"),
        math_high_count=("math_high_count", "sum"),
        if_only_count=("if_only_count", "sum"),
        math_only_count=("math_only_count", "sum"),
        both_high_count=("both_high_count", "sum"),
        union_high_count=("union_high_count", "sum"),
        parameter_count=("parameter", "count"),
    )
)

# Convert counts to coordinate-level ratios for shape-normalized comparison.
layer_overlap_df["if_high_ratio"] = layer_overlap_df["if_high_count"] / layer_overlap_df["numel"].clip(lower=1)
layer_overlap_df["math_high_ratio"] = layer_overlap_df["math_high_count"] / layer_overlap_df["numel"].clip(lower=1)
layer_overlap_df["if_only_ratio"] = layer_overlap_df["if_only_count"] / layer_overlap_df["numel"].clip(lower=1)
layer_overlap_df["math_only_ratio"] = layer_overlap_df["math_only_count"] / layer_overlap_df["numel"].clip(lower=1)
layer_overlap_df["both_high_ratio"] = layer_overlap_df["both_high_count"] / layer_overlap_df["numel"].clip(lower=1)
layer_overlap_df["jaccard"] = layer_overlap_df["both_high_count"] / layer_overlap_df["union_high_count"].clip(lower=1)
layer_overlap_df["layer_sort_key"] = layer_overlap_df["layer_bucket"].map(
    {label: idx for idx, label in enumerate(_sorted_layer_order(layer_overlap_df["layer_bucket"].tolist()))}
)
layer_overlap_df = layer_overlap_df.sort_values("layer_sort_key").drop(columns=["layer_sort_key"])
layer_overlap_df["layer_index"] = layer_overlap_df["layer_bucket"].apply(parse_layer_index)

layer_overlap_csv = analysis_dir / "importance_layer_overlap_stats.csv"
layer_overlap_df.to_csv(layer_overlap_csv, index=False)
print(f"Saved layer overlap statistics: {layer_overlap_csv}")

# Focus the histogram x-axis on true decoder layer indices.
decoder_layer_df = (
    layer_overlap_df[layer_overlap_df["layer_index"].notna()]
    .copy()
    .sort_values("layer_index")
)

if decoder_layer_df.empty:
    raise ValueError("No decoder layer buckets found. Check parameter naming convention.")

x = decoder_layer_df["layer_index"].astype(int).to_numpy()

# Histogram A: IF/Math/Both high-importance coordinate ratios by layer.
fig, ax = plt.subplots(figsize=(14.0, 4.5))
bar_width = 0.28
ax.bar(x - bar_width, decoder_layer_df["if_high_ratio"], width=bar_width, color="#1f77b4", label="if_high_ratio")
ax.bar(x, decoder_layer_df["math_high_ratio"], width=bar_width, color="#ff7f0e", label="math_high_ratio")
ax.bar(x + bar_width, decoder_layer_df["both_high_ratio"], width=bar_width, color="#2ca02c", label="both_high_ratio")
ax.set_xlabel("Decoder layer index")
ax.set_ylabel("High-importance coordinate ratio")
ax.set_title("High-Importance Location Histogram by Layer")
ax.set_xticks(x)
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
layer_hist_ratio_png = analysis_dir / "importance_layer_hist_high_ratio.png"
fig.savefig(layer_hist_ratio_png, dpi=220)
plt.show()
print(f"Saved layer histogram (ratio): {layer_hist_ratio_png}")

# Histogram B: stacked overlap composition by layer.
fig, ax = plt.subplots(figsize=(14.0, 4.5))
ax.bar(x, decoder_layer_df["both_high_ratio"], width=0.82, color="#2ca02c", label="both")
ax.bar(
    x,
    decoder_layer_df["if_only_ratio"],
    width=0.82,
    bottom=decoder_layer_df["both_high_ratio"],
    color="#1f77b4",
    label="if_only",
)
ax.bar(
    x,
    decoder_layer_df["math_only_ratio"],
    width=0.82,
    bottom=(decoder_layer_df["both_high_ratio"] + decoder_layer_df["if_only_ratio"]),
    color="#ff7f0e",
    label="math_only",
)
ax.set_xlabel("Decoder layer index")
ax.set_ylabel("Ratio within layer coordinates")
ax.set_title("Layer-wise High-Importance Overlap Composition")
ax.set_xticks(x)
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
layer_hist_stack_png = analysis_dir / "importance_layer_hist_overlap_stacked.png"
fig.savefig(layer_hist_stack_png, dpi=220)
plt.show()
print(f"Saved layer histogram (stacked): {layer_hist_stack_png}")

# Histogram C: number of parameters that have at least one high coordinate.
parameter_activity_df = overlap_df.copy()
parameter_activity_df["if_active_param"] = (parameter_activity_df["if_high_count"] > 0).astype(int)
parameter_activity_df["math_active_param"] = (parameter_activity_df["math_high_count"] > 0).astype(int)
parameter_activity_df["both_active_param"] = (
    (parameter_activity_df["if_high_count"] > 0) & (parameter_activity_df["math_high_count"] > 0)
).astype(int)

layer_parameter_activity_df = (
    parameter_activity_df.groupby("layer_bucket", as_index=False)
    .agg(
        if_active_param=("if_active_param", "sum"),
        math_active_param=("math_active_param", "sum"),
        both_active_param=("both_active_param", "sum"),
        parameter_count=("parameter", "count"),
    )
)
layer_parameter_activity_df["layer_index"] = layer_parameter_activity_df["layer_bucket"].apply(parse_layer_index)
layer_parameter_activity_df = (
    layer_parameter_activity_df[layer_parameter_activity_df["layer_index"].notna()]
    .copy()
    .sort_values("layer_index")
)

x_param = layer_parameter_activity_df["layer_index"].astype(int).to_numpy()
fig, ax = plt.subplots(figsize=(14.0, 4.5))
bar_width = 0.28
ax.bar(x_param - bar_width, layer_parameter_activity_df["if_active_param"], width=bar_width, color="#1f77b4", label="if_active_param")
ax.bar(x_param, layer_parameter_activity_df["math_active_param"], width=bar_width, color="#ff7f0e", label="math_active_param")
ax.bar(x_param + bar_width, layer_parameter_activity_df["both_active_param"], width=bar_width, color="#2ca02c", label="both_active_param")
ax.set_xlabel("Decoder layer index")
ax.set_ylabel("# parameters with high-importance coordinates")
ax.set_title("Parameter-level High-Importance Presence by Layer")
ax.set_xticks(x_param)
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
layer_hist_param_png = analysis_dir / "importance_layer_hist_parameter_presence.png"
fig.savefig(layer_hist_param_png, dpi=220)
plt.show()
print(f"Saved layer histogram (parameter presence): {layer_hist_param_png}")

# Non-decoder buckets are reported as a compact table for completeness.
non_decoder_df = layer_overlap_df[layer_overlap_df["layer_index"].isna()].copy()
if not non_decoder_df.empty:
    print("Non-decoder bucket summary:")
    display(
        non_decoder_df[
            [
                "layer_bucket",
                "numel",
                "if_high_ratio",
                "math_high_ratio",
                "both_high_ratio",
                "jaccard",
            ]
        ]
    )

# Keep the existing top-parameter overlap table for drill-down.
top_overlap_df = overlap_df.sort_values("both_high_ratio", ascending=False).head(25).copy()
display(
    top_overlap_df[
        [
            "parameter",
            "layer_bucket",
            "both_high_ratio",
            "if_high_ratio",
            "math_high_ratio",
            "jaccard",
        ]
    ]
)

print("Importance sparsity/overlap analysis finished (layer histograms).")


## Importance Distribution By Layer (IF vs Math)

This section inspects the actual value distribution of importance tensors. It compares IF vs Math globally and per decoder layer, and quantifies whether each layer is uniform across parameters or dominated by a few parameters.


In [ ]:
import math
from collections import defaultdict
import matplotlib.pyplot as plt


def infer_layer_bucket_for_distribution(parameter_name: str) -> str:
    """Map a parameter name to a layer bucket for distribution analysis.

    Args:
        parameter_name: Full parameter name from model `named_parameters()`.

    Returns:
        Bucket label. Decoder block tensors are represented as `layer_<idx>`.
    """

    layer_match = re.search(r"layers\.(\d+)\.", parameter_name)
    if layer_match is not None:
        layer_index = int(layer_match.group(1))
        return f"layer_{layer_index:02d}"

    if "embed_tokens" in parameter_name:
        return "embeddings"
    if "lm_head" in parameter_name:
        return "lm_head"
    if "norm" in parameter_name:
        return "final_norm"
    return "other"


def parse_layer_index_for_distribution(layer_bucket: str) -> int | None:
    """Extract decoder layer index from a bucket label.

    Args:
        layer_bucket: Bucket label such as `layer_00`.

    Returns:
        Integer layer index for decoder layers, otherwise `None`.
    """

    layer_match = re.match(r"layer_(\d+)$", layer_bucket)
    if layer_match is None:
        return None
    return int(layer_match.group(1))


def sample_tensor_values(
    tensor: torch.Tensor,
    max_samples: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """Sample coordinate values from a tensor without loading full copies.

    Args:
        tensor: Importance tensor on CPU.
        max_samples: Maximum number of sampled coordinates.
        rng: NumPy random generator for reproducibility.

    Returns:
        1D NumPy array of sampled values (float32).
    """

    flat = tensor.reshape(-1)
    numel = int(flat.numel())
    if numel == 0:
        return np.empty((0,), dtype=np.float32)

    if numel <= max_samples:
        return flat.detach().cpu().numpy().astype(np.float32, copy=False)

    # Randomly sample coordinates so distribution plots remain memory-efficient.
    sample_indices = rng.choice(numel, size=max_samples, replace=False)
    sample_index_tensor = torch.from_numpy(sample_indices).long()
    sampled = flat.index_select(0, sample_index_tensor)
    return sampled.detach().cpu().numpy().astype(np.float32, copy=False)


def gini_coefficient(values: np.ndarray) -> float:
    """Compute Gini coefficient for non-negative values.

    Args:
        values: 1D array of non-negative values.

    Returns:
        Gini coefficient in `[0, 1]`; larger means stronger concentration.
    """

    if values.size == 0:
        return 0.0

    clipped = np.clip(values.astype(np.float64, copy=False), a_min=0.0, a_max=None)
    total = clipped.sum()
    if total <= 0.0:
        return 0.0

    sorted_values = np.sort(clipped)
    n = sorted_values.size
    cumulative_index = np.arange(1, n + 1, dtype=np.float64)
    gini = (2.0 * np.sum(cumulative_index * sorted_values) / (n * total)) - (n + 1.0) / n
    return float(gini)


def summarize_quantiles(values: np.ndarray, quantiles: Sequence[float]) -> Dict[str, float]:
    """Compute robust quantiles for an array, returning NaN when empty.

    Args:
        values: 1D array.
        quantiles: Quantiles in `[0, 1]`.

    Returns:
        Dictionary keyed by `qXX` (e.g., `q50`).
    """

    if values.size == 0:
        return {f"q{int(q * 100):02d}": float("nan") for q in quantiles}

    output: Dict[str, float] = {}
    for q in quantiles:
        output[f"q{int(q * 100):02d}"] = float(np.quantile(values, q))
    return output


# Ensure importance tensors are available even if this cell is run independently.
if "importance_by_task" not in globals() or not importance_by_task:
    if "importance_paths" not in globals() or not importance_paths:
        raise RuntimeError(
            "importance_by_task and importance_paths are both unavailable. "
            "Run attribution first to generate importance tensors."
        )
    importance_by_task = {
        task_name: torch.load(path, map_location="cpu")
        for task_name, path in importance_paths.items()
    }

analysis_dir = RUNTIME.output_root / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

task_names = sorted(list(importance_by_task.keys()))
if set(task_names) != {"if", "math"}:
    raise ValueError(f"Expected exactly tasks {{'if', 'math'}}, but got {task_names}")

# -------------------------------------------------------------------------
# A) Parameter-level summary statistics (full tensors, no sampling)
# -------------------------------------------------------------------------
parameter_rows: List[Dict[str, Any]] = []

for task_name in task_names:
    for parameter_name, tensor in tqdm(
        importance_by_task[task_name].items(),
        desc=f"Parameter summary ({task_name})",
    ):
        tensor_fp32 = tensor.to(torch.float32)
        numel = int(tensor_fp32.numel())
        nonzero_count = int((tensor_fp32 > 0).sum().item())

        parameter_rows.append(
            {
                "task": task_name,
                "parameter": parameter_name,
                "layer_bucket": infer_layer_bucket_for_distribution(parameter_name),
                "layer_index": parse_layer_index_for_distribution(
                    infer_layer_bucket_for_distribution(parameter_name)
                ),
                "numel": numel,
                "sum_importance": float(tensor_fp32.sum().item()),
                "mean_importance": float(tensor_fp32.mean().item()),
                "max_importance": float(tensor_fp32.max().item()),
                "nonzero_count": nonzero_count,
                "nonzero_ratio": nonzero_count / max(numel, 1),
            }
        )

parameter_distribution_df = pd.DataFrame(parameter_rows)
parameter_distribution_csv = analysis_dir / "importance_distribution_parameter_stats.csv"
parameter_distribution_df.to_csv(parameter_distribution_csv, index=False)
print(f"Saved parameter distribution stats: {parameter_distribution_csv}")

# -------------------------------------------------------------------------
# B) Sampled coordinate distributions (global + layer-wise)
# -------------------------------------------------------------------------
max_samples_per_parameter = 4096
max_samples_per_layer = 200_000
sampling_seed = RUNTIME.seed
rng = np.random.default_rng(sampling_seed)

layer_samples_by_task: Dict[str, Dict[str, np.ndarray]] = {}
global_samples_by_task: Dict[str, np.ndarray] = {}

for task_name in task_names:
    sampled_by_layer_lists: Dict[str, List[np.ndarray]] = defaultdict(list)

    for parameter_name, tensor in tqdm(
        importance_by_task[task_name].items(),
        desc=f"Sample coordinates ({task_name})",
    ):
        layer_bucket = infer_layer_bucket_for_distribution(parameter_name)
        sampled_values = sample_tensor_values(
            tensor=tensor,
            max_samples=max_samples_per_parameter,
            rng=rng,
        )
        sampled_by_layer_lists[layer_bucket].append(sampled_values)

    sampled_by_layer: Dict[str, np.ndarray] = {}
    for layer_bucket, chunks in sampled_by_layer_lists.items():
        if not chunks:
            sampled_by_layer[layer_bucket] = np.empty((0,), dtype=np.float32)
            continue

        concatenated = np.concatenate(chunks, axis=0).astype(np.float32, copy=False)
        if concatenated.size > max_samples_per_layer:
            keep_indices = rng.choice(concatenated.size, size=max_samples_per_layer, replace=False)
            concatenated = concatenated[keep_indices]

        sampled_by_layer[layer_bucket] = concatenated

    layer_samples_by_task[task_name] = sampled_by_layer
    global_samples_by_task[task_name] = np.concatenate(list(sampled_by_layer.values()), axis=0)

# -------------------------------------------------------------------------
# C) Global histogram comparison (IF vs Math)
# -------------------------------------------------------------------------
global_summary_rows: List[Dict[str, Any]] = []
for task_name in task_names:
    sampled = global_samples_by_task[task_name]
    nonzero = sampled[sampled > 0]
    zero_ratio = float(1.0 - (nonzero.size / max(sampled.size, 1)))

    quantile_stats = summarize_quantiles(nonzero, quantiles=[0.5, 0.9, 0.99])
    global_summary_rows.append(
        {
            "task": task_name,
            "sampled_count": int(sampled.size),
            "nonzero_count": int(nonzero.size),
            "zero_ratio": zero_ratio,
            "q50": quantile_stats["q50"],
            "q90": quantile_stats["q90"],
            "q99": quantile_stats["q99"],
            "max": float(nonzero.max()) if nonzero.size > 0 else float("nan"),
        }
    )

global_distribution_df = pd.DataFrame(global_summary_rows)
global_distribution_csv = analysis_dir / "importance_distribution_global_summary.csv"
global_distribution_df.to_csv(global_distribution_csv, index=False)
print(f"Saved global distribution summary: {global_distribution_csv}")
display(global_distribution_df)

# Plot histogram on log10 scale using nonzero values only; zeros are reported separately.
fig, ax = plt.subplots(figsize=(9.0, 4.6))
for task_name, color in [("if", "#1f77b4"), ("math", "#ff7f0e")]:
    nonzero = global_samples_by_task[task_name][global_samples_by_task[task_name] > 0]
    if nonzero.size == 0:
        continue

    log_values = np.log10(nonzero)
    ax.hist(
        log_values,
        bins=80,
        density=True,
        alpha=0.45,
        color=color,
        label=f"{task_name} (nonzero)",
    )

ax.set_title("Global Importance Value Distribution (log10, nonzero coordinates)")
ax.set_xlabel("log10(importance)")
ax.set_ylabel("Density")
ax.legend(loc="upper left")
ax.grid(axis="y", alpha=0.25)

# Annotate zero-ratio gap, which is critical for interpreting sparsity.
annotation_lines = []
for _, row in global_distribution_df.iterrows():
    annotation_lines.append(f"{row['task']}: zero_ratio={row['zero_ratio']:.2%}")
ax.text(
    0.99,
    0.98,
    "\n".join(annotation_lines),
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "#cccccc"},
)

fig.tight_layout()
global_hist_png = analysis_dir / "importance_distribution_global_hist_log10.png"
fig.savefig(global_hist_png, dpi=220)
plt.show()
print(f"Saved global distribution histogram: {global_hist_png}")

# -------------------------------------------------------------------------
# D) Layer-wise histograms (decoder layers only, IF vs Math overlay)
# -------------------------------------------------------------------------
decoder_layer_buckets = sorted(
    {
        bucket
        for task_name in task_names
        for bucket in layer_samples_by_task[task_name].keys()
        if parse_layer_index_for_distribution(bucket) is not None
    },
    key=lambda x: parse_layer_index_for_distribution(x),
)

if not decoder_layer_buckets:
    raise ValueError("No decoder layers detected in importance tensor names.")

# Pre-compute layer-level zero ratios from full parameter summaries.
layer_zero_ratio_map: Dict[tuple[str, str], float] = {}
for task_name in task_names:
    task_df = parameter_distribution_df[parameter_distribution_df["task"] == task_name].copy()
    grouped = task_df.groupby("layer_bucket", as_index=False).agg(
        nonzero_count=("nonzero_count", "sum"),
        numel=("numel", "sum"),
    )
    grouped["zero_ratio"] = 1.0 - (grouped["nonzero_count"] / grouped["numel"].clip(lower=1))
    for _, row in grouped.iterrows():
        layer_zero_ratio_map[(task_name, row["layer_bucket"])] = float(row["zero_ratio"])

# Build common histogram bins from all sampled nonzero log-values for fair comparison.
all_log_values: List[np.ndarray] = []
for task_name in task_names:
    for layer_bucket in decoder_layer_buckets:
        sampled = layer_samples_by_task[task_name].get(layer_bucket, np.empty((0,), dtype=np.float32))
        nonzero = sampled[sampled > 0]
        if nonzero.size > 0:
            all_log_values.append(np.log10(nonzero))

if all_log_values:
    pooled_log = np.concatenate(all_log_values, axis=0)
    lower = float(np.quantile(pooled_log, 0.001))
    upper = float(np.quantile(pooled_log, 0.999))
    if not np.isfinite(lower) or not np.isfinite(upper) or lower >= upper:
        lower, upper = -12.0, -2.0
else:
    lower, upper = -12.0, -2.0

hist_bins = np.linspace(lower, upper, 60)

num_layers = len(decoder_layer_buckets)
num_cols = 4
num_rows = int(math.ceil(num_layers / num_cols))
fig, axes = plt.subplots(num_rows, num_cols, figsize=(18.0, max(3.0 * num_rows, 6.0)), sharex=True, sharey=True)
axes = np.array(axes).reshape(num_rows, num_cols)

for idx, layer_bucket in enumerate(decoder_layer_buckets):
    row = idx // num_cols
    col = idx % num_cols
    ax = axes[row, col]

    for task_name, color in [("if", "#1f77b4"), ("math", "#ff7f0e")]:
        sampled = layer_samples_by_task[task_name].get(layer_bucket, np.empty((0,), dtype=np.float32))
        nonzero = sampled[sampled > 0]
        if nonzero.size == 0:
            continue

        log_values = np.log10(nonzero)
        ax.hist(log_values, bins=hist_bins, density=True, alpha=0.40, color=color)

    layer_idx = parse_layer_index_for_distribution(layer_bucket)
    if_zero_ratio = layer_zero_ratio_map.get(("if", layer_bucket), float("nan"))
    math_zero_ratio = layer_zero_ratio_map.get(("math", layer_bucket), float("nan"))
    ax.set_title(
        f"L{layer_idx:02d} | z(if)={if_zero_ratio:.2%}, z(math)={math_zero_ratio:.2%}",
        fontsize=9,
    )
    ax.grid(axis="y", alpha=0.20)

# Hide empty axes when layer count is not divisible by column count.
for idx in range(num_layers, num_rows * num_cols):
    row = idx // num_cols
    col = idx % num_cols
    axes[row, col].axis("off")

fig.suptitle("Layer-wise Importance Distribution (log10, nonzero coordinates)", fontsize=14)
fig.supxlabel("log10(importance)")
fig.supylabel("Density")

legend_handles = [
    plt.Line2D([0], [0], color="#1f77b4", lw=6, alpha=0.5, label="if"),
    plt.Line2D([0], [0], color="#ff7f0e", lw=6, alpha=0.5, label="math"),
]
fig.legend(handles=legend_handles, loc="upper right")
fig.tight_layout(rect=[0.0, 0.02, 1.0, 0.95])

layer_hist_grid_png = analysis_dir / "importance_distribution_layer_hist_grid.png"
fig.savefig(layer_hist_grid_png, dpi=220)
plt.show()
print(f"Saved layer histogram grid: {layer_hist_grid_png}")

# -------------------------------------------------------------------------
# E) Layer range and concentration analysis
# -------------------------------------------------------------------------
layer_stats_rows: List[Dict[str, Any]] = []
top_parameter_rows: List[Dict[str, Any]] = []

for task_name in task_names:
    task_df = parameter_distribution_df[parameter_distribution_df["task"] == task_name].copy()

    for layer_bucket, layer_param_df in task_df.groupby("layer_bucket"):
        layer_index = parse_layer_index_for_distribution(layer_bucket)

        sampled = layer_samples_by_task[task_name].get(layer_bucket, np.empty((0,), dtype=np.float32))
        nonzero = sampled[sampled > 0]
        log_nonzero = np.log10(nonzero) if nonzero.size > 0 else np.empty((0,), dtype=np.float32)

        # Quantiles summarize the effective value range inside each layer.
        quantiles_raw = summarize_quantiles(nonzero, quantiles=[0.5, 0.9, 0.99])
        quantiles_log = summarize_quantiles(log_nonzero, quantiles=[0.5, 0.9, 0.99])

        param_sums = layer_param_df["sum_importance"].to_numpy(dtype=np.float64)
        layer_total_sum = float(param_sums.sum())
        top1_share = float(np.max(param_sums) / layer_total_sum) if layer_total_sum > 0 else 0.0

        if param_sums.size >= 3 and layer_total_sum > 0:
            top3_share = float(np.sort(param_sums)[-3:].sum() / layer_total_sum)
        else:
            top3_share = float(np.sum(param_sums) / layer_total_sum) if layer_total_sum > 0 else 0.0

        layer_stats_rows.append(
            {
                "task": task_name,
                "layer_bucket": layer_bucket,
                "layer_index": layer_index,
                "num_parameters": int(len(layer_param_df)),
                "numel": int(layer_param_df["numel"].sum()),
                "zero_ratio": 1.0 - float(layer_param_df["nonzero_count"].sum() / layer_param_df["numel"].sum()),
                "sum_importance": layer_total_sum,
                "top1_param_share": top1_share,
                "top3_param_share": top3_share,
                "gini_param_sum": gini_coefficient(param_sums),
                "raw_q50": quantiles_raw["q50"],
                "raw_q90": quantiles_raw["q90"],
                "raw_q99": quantiles_raw["q99"],
                "raw_max": float(nonzero.max()) if nonzero.size > 0 else float("nan"),
                "log_q50": quantiles_log["q50"],
                "log_q90": quantiles_log["q90"],
                "log_q99": quantiles_log["q99"],
                "log_max": float(log_nonzero.max()) if log_nonzero.size > 0 else float("nan"),
            }
        )

        # Save top parameters for direct inspection of layer-level outliers.
        layer_param_df_sorted = layer_param_df.sort_values("sum_importance", ascending=False).copy()
        cumulative = layer_param_df_sorted["sum_importance"].cumsum()
        denominator = max(layer_total_sum, 1e-30)

        for rank, (_, row) in enumerate(layer_param_df_sorted.head(10).iterrows(), start=1):
            top_parameter_rows.append(
                {
                    "task": task_name,
                    "layer_bucket": layer_bucket,
                    "layer_index": layer_index,
                    "rank": rank,
                    "parameter": row["parameter"],
                    "sum_importance": float(row["sum_importance"]),
                    "share_of_layer_sum": float(row["sum_importance"] / denominator),
                    "cumulative_share": float(cumulative.iloc[rank - 1] / denominator),
                    "nonzero_ratio": float(row["nonzero_ratio"]),
                    "max_importance": float(row["max_importance"]),
                }
            )

layer_distribution_df = pd.DataFrame(layer_stats_rows)
layer_distribution_csv = analysis_dir / "importance_distribution_layer_stats.csv"
layer_distribution_df.to_csv(layer_distribution_csv, index=False)
print(f"Saved layer distribution stats: {layer_distribution_csv}")

top_parameter_df = pd.DataFrame(top_parameter_rows)
top_parameter_csv = analysis_dir / "importance_distribution_layer_top_parameters.csv"
top_parameter_df.to_csv(top_parameter_csv, index=False)
print(f"Saved top-parameter concentration table: {top_parameter_csv}")

# Focus plots on decoder layers for clean x-axis ordering.
decoder_layer_stats_df = (
    layer_distribution_df[layer_distribution_df["layer_index"].notna()]
    .copy()
    .sort_values(["task", "layer_index"])
)

# Range plot: how value scale changes by layer.
fig, axes = plt.subplots(1, 2, figsize=(16.0, 4.8), sharey=True)
for axis_index, task_name in enumerate(task_names):
    ax = axes[axis_index]
    task_layer_df = decoder_layer_stats_df[decoder_layer_stats_df["task"] == task_name].copy()
    x = task_layer_df["layer_index"].astype(int).to_numpy()

    ax.plot(x, task_layer_df["log_q50"], marker="o", ms=3, lw=1.5, label="log_q50")
    ax.plot(x, task_layer_df["log_q90"], marker="o", ms=3, lw=1.5, label="log_q90")
    ax.plot(x, task_layer_df["log_q99"], marker="o", ms=3, lw=1.5, label="log_q99")
    ax.plot(x, task_layer_df["log_max"], marker="o", ms=3, lw=1.0, alpha=0.8, label="log_max")

    ax.set_title(f"{task_name}: Layer-wise Value Range")
    ax.set_xlabel("Decoder layer index")
    ax.set_ylabel("log10(importance)")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="best", fontsize=8)

fig.tight_layout()
layer_range_png = analysis_dir / "importance_distribution_layer_range_log10.png"
fig.savefig(layer_range_png, dpi=220)
plt.show()
print(f"Saved layer range plot: {layer_range_png}")

# Concentration plot: detect whether few parameters dominate each layer.
concentration_df = decoder_layer_stats_df.copy()
if_df = concentration_df[concentration_df["task"] == "if"].copy().sort_values("layer_index")
math_df = concentration_df[concentration_df["task"] == "math"].copy().sort_values("layer_index")

x = if_df["layer_index"].astype(int).to_numpy()
bar_width = 0.38

fig, axes = plt.subplots(2, 1, figsize=(14.0, 8.0), sharex=True)

# Top-1 share near 1.0 means one parameter dominates the layer.
axes[0].bar(x - bar_width / 2, if_df["top1_param_share"], width=bar_width, color="#1f77b4", label="if")
axes[0].bar(x + bar_width / 2, math_df["top1_param_share"], width=bar_width, color="#ff7f0e", label="math")
axes[0].set_ylabel("Top-1 parameter share")
axes[0].set_title("Layer-wise Parameter Concentration (Top-1 Share)")
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend(loc="upper right")

# Gini close to 0 means uniform; close to 1 means highly concentrated.
axes[1].plot(x, if_df["gini_param_sum"], marker="o", ms=3, lw=1.5, color="#1f77b4", label="if")
axes[1].plot(x, math_df["gini_param_sum"], marker="o", ms=3, lw=1.5, color="#ff7f0e", label="math")
axes[1].set_xlabel("Decoder layer index")
axes[1].set_ylabel("Gini over parameter sums")
axes[1].set_title("Layer-wise Parameter Concentration (Gini)")
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(loc="upper right")

fig.tight_layout()
layer_concentration_png = analysis_dir / "importance_distribution_layer_concentration.png"
fig.savefig(layer_concentration_png, dpi=220)
plt.show()
print(f"Saved layer concentration plot: {layer_concentration_png}")

# Quick diagnostic table: top parameters in first few decoder layers.
preview_df = top_parameter_df[top_parameter_df["layer_index"].notna()].copy()
preview_df = preview_df.sort_values(["task", "layer_index", "rank"]).head(40)
display(
    preview_df[
        [
            "task",
            "layer_index",
            "rank",
            "parameter",
            "share_of_layer_sum",
            "cumulative_share",
            "nonzero_ratio",
            "max_importance",
        ]
    ]
)

print("Importance distribution analysis finished (global + layer-wise histograms, range, concentration).")


## JWCM Importance Concentration Diagnostics

This section mirrors Fisher concentration diagnostics, but uses saved JWCM importance tensors (`S_j`) instead.

Goal:
- quantify whether JWCM importance mass is concentrated in a small subset of coordinates/tensors,
- compare IF vs Math concentration behavior using Gini/Lorenz/top-k mass summaries.


In [ ]:
# JWCM importance concentration diagnostics
# This cell mirrors Fisher concentration diagnostics but uses JWCM importance tensors.
# It quantifies whether JWCM scale is concentrated in a small subset of parameters
# (sparse/highly skewed) or spread relatively uniformly across the model.

import json
from pathlib import Path
from typing import Any, Dict, Mapping, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch


def resolve_importance_paths_from_context(runtime_output_root: Path) -> Dict[str, Path]:
    """Resolve task->importance tensor paths from in-memory state or run summary.

    Why this exists:
        This notebook is often run in partial/restart mode. In that case,
        `importance_paths` may be missing from memory. We recover paths from
        metadata JSON so this diagnostics cell can run independently.

    Args:
        runtime_output_root: Runtime root that contains `metadata/jwcm_v2_run_summary.json`.

    Returns:
        Mapping from task name to on-disk importance `.pt` path.

    Raises:
        RuntimeError: If no valid path mapping can be resolved.
    """

    # Preferred source: in-memory variable produced by attribution stage.
    if 'importance_paths' in globals():
        candidate_paths = globals().get('importance_paths')
        if isinstance(candidate_paths, Mapping) and len(candidate_paths) > 0:
            return {str(task): Path(str(path)) for task, path in candidate_paths.items()}

    # Fallback source: persisted run summary metadata.
    summary_path = runtime_output_root / 'metadata' / 'jwcm_v2_run_summary.json'
    if summary_path.exists():
        summary_payload = json.loads(summary_path.read_text(encoding='utf-8'))
        summary_paths = summary_payload.get('importance_paths', {})
        if isinstance(summary_paths, Mapping) and len(summary_paths) > 0:
            return {str(task): Path(str(path)) for task, path in summary_paths.items()}

    raise RuntimeError(
        'Unable to resolve importance tensor paths. '
        'Run attribution stage first, or ensure metadata/jwcm_v2_run_summary.json exists.'
    )


def load_importance_dictionary(task_name: str, importance_path: Path) -> Mapping[str, torch.Tensor]:
    """Load one task's JWCM importance dictionary from disk.

    Args:
        task_name: Short task key used for readable error messages.
        importance_path: Path to serialized importance tensor dictionary (`.pt`).

    Returns:
        Parameter-name keyed importance dictionary on CPU.

    Raises:
        FileNotFoundError: If the expected importance file is missing.
        TypeError: If the deserialized object is not a mapping.
    """

    if not importance_path.exists():
        raise FileNotFoundError(
            f"Importance file for task '{task_name}' was not found: {importance_path}"
        )

    importance_object = torch.load(importance_path, map_location='cpu')
    if not isinstance(importance_object, Mapping):
        raise TypeError(
            f"Importance file for task '{task_name}' must contain a mapping, got {type(importance_object)}"
        )

    return importance_object


def collect_tensor_mass_stats(
    importance_tensors: Mapping[str, torch.Tensor],
) -> Tuple[pd.DataFrame, int, float]:
    """Collect per-parameter importance mass statistics.

    Why this exists:
        Tensor-level mass share reveals whether a small number of named
        parameter tensors dominate total JWCM importance mass.

    Args:
        importance_tensors: Mapping from parameter name to importance tensor.

    Returns:
        Tuple of:
        - DataFrame with per-parameter stats (`numel`, `mass`, `mean`, `max`, `mass_share`)
        - Total element count across all tensors
        - Total importance mass across all tensors
    """

    rows = []
    total_numel = 0
    total_mass = 0.0

    for parameter_name, importance_tensor in importance_tensors.items():
        # Use absolute value defensively in case small negative values appear
        # due to numeric artifacts. Theoretical JWCM importance is non-negative.
        flat_values = importance_tensor.detach().to(torch.float32).abs().reshape(-1)

        numel = int(flat_values.numel())
        mass = float(flat_values.sum().item())
        mean_value = float(mass / max(numel, 1))
        max_value = float(flat_values.max().item()) if numel > 0 else 0.0

        rows.append(
            {
                'parameter_name': parameter_name,
                'numel': numel,
                'mass': mass,
                'mean': mean_value,
                'max': max_value,
            }
        )

        total_numel += numel
        total_mass += mass

    tensor_stats_df = pd.DataFrame(rows)
    if total_mass > 0.0 and not tensor_stats_df.empty:
        tensor_stats_df['mass_share'] = tensor_stats_df['mass'] / total_mass
    else:
        tensor_stats_df['mass_share'] = 0.0

    return tensor_stats_df, total_numel, total_mass


def sample_importance_values(
    importance_tensors: Mapping[str, torch.Tensor],
    total_numel: int,
    sample_size: int,
    seed: int,
) -> np.ndarray:
    """Uniformly sample importance elements without materializing one giant vector.

    Why this exists:
        Full flattening across all model parameters is memory-heavy for large LMs.
        This two-pass index strategy gives an unbiased global sample with low memory.

    Args:
        importance_tensors: Mapping from parameter name to importance tensor.
        total_numel: Total scalar element count across all tensors.
        sample_size: Number of scalar importance values to sample.
        seed: RNG seed for reproducibility.

    Returns:
        1D NumPy array of sampled non-negative importance values.
    """

    if total_numel <= 0 or sample_size <= 0:
        return np.zeros(0, dtype=np.float32)

    effective_sample_size = int(min(sample_size, total_numel))
    rng = np.random.default_rng(seed=seed)

    # Sampling with replacement keeps memory predictable for huge models.
    sampled_global_indices = np.sort(
        rng.integers(low=0, high=total_numel, size=effective_sample_size, dtype=np.int64)
    )

    sampled_values = np.empty(effective_sample_size, dtype=np.float32)
    write_cursor = 0
    tensor_offset = 0

    for importance_tensor in importance_tensors.values():
        flat_values = importance_tensor.detach().to(torch.float32).abs().reshape(-1)
        tensor_numel = int(flat_values.numel())

        # Identify sampled indices that belong to current tensor slice.
        left = np.searchsorted(sampled_global_indices, tensor_offset, side='left')
        right = np.searchsorted(sampled_global_indices, tensor_offset + tensor_numel, side='left')

        if right > left:
            local_indices = sampled_global_indices[left:right] - tensor_offset
            local_index_tensor = torch.from_numpy(local_indices.astype(np.int64))
            local_values = flat_values.index_select(dim=0, index=local_index_tensor)

            next_cursor = write_cursor + (right - left)
            sampled_values[write_cursor:next_cursor] = local_values.cpu().numpy()
            write_cursor = next_cursor

        tensor_offset += tensor_numel

    if write_cursor != effective_sample_size:
        raise RuntimeError(
            f"Sampling bookkeeping mismatch: expected {effective_sample_size}, got {write_cursor}"
        )

    return sampled_values


def compute_lorenz_curve(values: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Compute Lorenz curve points for non-negative values.

    Args:
        values: 1D array of importance values.

    Returns:
        Tuple (`population_share`, `mass_share`) for Lorenz plotting.
    """

    if values.size == 0:
        return np.array([0.0, 1.0]), np.array([0.0, 1.0])

    clipped_values = np.clip(values.astype(np.float64), a_min=0.0, a_max=None)
    sorted_values = np.sort(clipped_values)
    cumulative_mass = np.cumsum(sorted_values)

    total_mass = float(cumulative_mass[-1])
    if total_mass <= 0.0:
        return np.array([0.0, 1.0]), np.array([0.0, 1.0])

    mass_share = np.concatenate(([0.0], cumulative_mass / total_mass))
    population_share = np.linspace(0.0, 1.0, mass_share.size)
    return population_share, mass_share


def compute_gini_from_lorenz(population_share: np.ndarray, mass_share: np.ndarray) -> float:
    """Compute Gini coefficient from Lorenz curve points.

    Args:
        population_share: X-axis values of Lorenz curve.
        mass_share: Y-axis values of Lorenz curve.

    Returns:
        Gini coefficient in [0, 1], where larger means stronger concentration.
    """

    area_under_curve = float(np.trapz(mass_share, population_share))
    gini_value = 1.0 - 2.0 * area_under_curve
    return float(np.clip(gini_value, 0.0, 1.0))


def compute_top_mass_share(values: np.ndarray, top_fraction: float) -> float:
    """Compute mass captured by top-ranked fraction of sampled values.

    Args:
        values: Sampled importance values.
        top_fraction: Fraction in (0, 1], e.g., 0.01 for top 1%.

    Returns:
        Fraction of total sampled mass carried by top-ranked elements.
    """

    if values.size == 0:
        return 0.0

    clipped_values = np.clip(values.astype(np.float64), a_min=0.0, a_max=None)
    total_mass = float(clipped_values.sum())
    if total_mass <= 0.0:
        return 0.0

    top_count = int(max(1, np.ceil(clipped_values.size * float(top_fraction))))
    top_values = np.partition(clipped_values, clipped_values.size - top_count)[-top_count:]
    return float(top_values.sum() / total_mass)


SAMPLE_SIZE_PER_TASK = 500_000
TOP_FRACTIONS = (0.001, 0.01, 0.05, 0.10)
EPSILON_FOR_LOG = 1e-30

importance_paths_by_task = resolve_importance_paths_from_context(runtime_output_root=RUNTIME.output_root)
print('Resolved importance paths:')
for task_name, importance_path in sorted(importance_paths_by_task.items()):
    print(f"  - {task_name}: {importance_path}")


# Build per-task diagnostics payload for shared plotting and tabular summaries.
task_diagnostics: Dict[str, Dict[str, Any]] = {}

for task_spec in TASK_SPECS:
    task_name = task_spec.name
    if task_name not in importance_paths_by_task:
        print(f"[warn] Missing importance path for task={task_name}; skipping.")
        continue

    importance_path = Path(importance_paths_by_task[task_name])

    importance_tensors = load_importance_dictionary(task_name=task_name, importance_path=importance_path)
    tensor_stats_df, total_numel, total_mass = collect_tensor_mass_stats(
        importance_tensors=importance_tensors,
    )
    sampled_values = sample_importance_values(
        importance_tensors=importance_tensors,
        total_numel=total_numel,
        sample_size=SAMPLE_SIZE_PER_TASK,
        seed=RUNTIME.seed,
    )

    population_share, mass_share = compute_lorenz_curve(sampled_values)

    task_diagnostics[task_name] = {
        'tensor_stats_df': tensor_stats_df,
        'total_numel': total_numel,
        'total_mass': total_mass,
        'sampled_values': sampled_values,
        'population_share': population_share,
        'mass_share': mass_share,
    }

if not task_diagnostics:
    raise RuntimeError('No task diagnostics were built. Check importance paths and saved artifacts.')

# Build compact numeric summary for side-by-side concentration comparison.
summary_rows = []
for task_name, diagnostic in task_diagnostics.items():
    sampled_values = diagnostic['sampled_values']
    tensor_stats_df = diagnostic['tensor_stats_df']

    top_tensor_share = (
        float(tensor_stats_df['mass_share'].max()) if not tensor_stats_df.empty else 0.0
    )
    top10_tensor_share = (
        float(tensor_stats_df['mass_share'].nlargest(min(10, len(tensor_stats_df))).sum())
        if not tensor_stats_df.empty
        else 0.0
    )

    summary_row = {
        'task': task_name,
        'total_elements': int(diagnostic['total_numel']),
        'total_importance_mass': float(diagnostic['total_mass']),
        'sample_size': int(sampled_values.size),
        'sample_fraction': float(sampled_values.size / max(diagnostic['total_numel'], 1)),
        'gini_sampled': compute_gini_from_lorenz(
            population_share=diagnostic['population_share'],
            mass_share=diagnostic['mass_share'],
        ),
        'top_tensor_mass_share': top_tensor_share,
        'top10_tensor_mass_share': top10_tensor_share,
    }

    for fraction in TOP_FRACTIONS:
        key = f"top_{fraction * 100:.1f}pct_mass_share".replace('.', '_')
        summary_row[key] = compute_top_mass_share(sampled_values, top_fraction=fraction)

    summary_rows.append(summary_row)

summary_df = pd.DataFrame(summary_rows).sort_values('task').reset_index(drop=True)
print('Interpretation hint: higher Gini / higher top-x% mass share means stronger concentration (sparser JWCM scale).')
display(summary_df)

# 2x2 panel to compare element-level and tensor-level concentration patterns.
figure, axes = plt.subplots(2, 2, figsize=(18, 12))
ax_hist, ax_lorenz, ax_topk, ax_tensor_cum = axes.flatten()

for task_name, diagnostic in task_diagnostics.items():
    sampled_values = diagnostic['sampled_values']
    log_values = np.log10(np.clip(sampled_values, a_min=EPSILON_FOR_LOG, a_max=None))

    ax_hist.hist(
        log_values,
        bins=120,
        density=True,
        alpha=0.45,
        label=task_name,
    )

ax_hist.set_title('Sampled JWCM importance distribution (log10 scale)')
ax_hist.set_xlabel('log10(importance + epsilon)')
ax_hist.set_ylabel('Density')
ax_hist.legend()

for task_name, diagnostic in task_diagnostics.items():
    ax_lorenz.plot(
        diagnostic['population_share'],
        diagnostic['mass_share'],
        linewidth=2.0,
        label=task_name,
    )

ax_lorenz.plot([0.0, 1.0], [0.0, 1.0], 'k--', linewidth=1.0, label='Uniform baseline')
ax_lorenz.set_title('Lorenz curve of sampled JWCM importance mass')
ax_lorenz.set_xlabel('Fraction of parameters (sampled)')
ax_lorenz.set_ylabel('Cumulative importance mass fraction')
ax_lorenz.legend()

fraction_labels = [f"top {fraction * 100:.1f}%" for fraction in TOP_FRACTIONS]
bar_positions = np.arange(len(fraction_labels), dtype=np.float64)
task_names = list(task_diagnostics.keys())
bar_width = 0.8 / max(len(task_names), 1)

for task_index, task_name in enumerate(task_names):
    sampled_values = task_diagnostics[task_name]['sampled_values']
    heights = [
        compute_top_mass_share(sampled_values, top_fraction=fraction)
        for fraction in TOP_FRACTIONS
    ]

    offset = (task_index - (len(task_names) - 1) / 2.0) * bar_width
    ax_topk.bar(bar_positions + offset, heights, width=bar_width, label=task_name)

ax_topk.set_xticks(bar_positions)
ax_topk.set_xticklabels(fraction_labels, rotation=0)
ax_topk.set_ylim(0.0, 1.0)
ax_topk.set_title('Mass captured by top-ranked sampled importance elements')
ax_topk.set_ylabel('Importance mass fraction')
ax_topk.legend()

for task_name, diagnostic in task_diagnostics.items():
    tensor_stats_df = diagnostic['tensor_stats_df'].sort_values('mass_share', ascending=False)
    if tensor_stats_df.empty:
        continue

    cumulative_mass = tensor_stats_df['mass_share'].cumsum().to_numpy()
    tensor_fraction = (
        np.arange(1, cumulative_mass.size + 1, dtype=np.float64) / cumulative_mass.size
    )
    ax_tensor_cum.plot(tensor_fraction, cumulative_mass, linewidth=2.0, label=task_name)

ax_tensor_cum.plot([0.0, 1.0], [0.0, 1.0], 'k--', linewidth=1.0, label='Uniform baseline')
ax_tensor_cum.set_title('Cumulative mass over parameter tensors')
ax_tensor_cum.set_xlabel('Fraction of parameter tensors (sorted by mass)')
ax_tensor_cum.set_ylabel('Cumulative importance mass fraction')
ax_tensor_cum.legend()

plt.tight_layout()
plt.show()

# Print top tensor contributors for quick qualitative inspection.
for task_name, diagnostic in task_diagnostics.items():
    print(f"\nTop 15 parameter tensors by importance mass share - task={task_name}")
    display(
        diagnostic['tensor_stats_df']
        .sort_values('mass_share', ascending=False)
        .head(15)
        [['parameter_name', 'numel', 'mass_share', 'mean', 'max']]
        .reset_index(drop=True)
    )


In [ ]:
# JWCM importance dominant layer/head analysis
# This cell localizes which layers, module groups, and attention heads dominate
# JWCM importance mass when global concentration metrics are extreme.

import json
import re
from pathlib import Path
from typing import Dict, Mapping, Optional, Tuple

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoConfig


LAYER_PATTERN = re.compile(r"^model\.layers\.(\d+)\.")
ATTN_QKV_WEIGHT_PATTERN = re.compile(
    r"^model\.layers\.(\d+)\.self_attn\.(q_proj|k_proj|v_proj)\.weight$"
)


def load_importance_dictionary_for_task(
    task_name: str,
    importance_path: Path,
) -> Mapping[str, torch.Tensor]:
    """Load one task importance dictionary from disk.

    Args:
        task_name: Task key used for diagnostics.
        importance_path: Path to serialized importance `.pt` mapping.

    Returns:
        Mapping from parameter name to importance tensor.

    Raises:
        FileNotFoundError: If artifact is missing.
        TypeError: If loaded object is not a mapping.
    """

    if not importance_path.exists():
        raise FileNotFoundError(
            f"Missing importance file for task '{task_name}': {importance_path}"
        )

    importance_obj = torch.load(importance_path, map_location='cpu')
    if not isinstance(importance_obj, Mapping):
        raise TypeError(
            f"Importance object for task '{task_name}' must be a mapping, got {type(importance_obj)}"
        )

    return importance_obj


def extract_layer_index(parameter_name: str) -> Optional[int]:
    """Extract transformer layer index from parameter name.

    Args:
        parameter_name: Named parameter key in Hugging Face model format.

    Returns:
        Integer layer index for keys in `model.layers.{idx}.*`, else `None`.
    """

    match = LAYER_PATTERN.match(parameter_name)
    if match is None:
        return None
    return int(match.group(1))


def classify_module_group(parameter_name: str) -> str:
    """Classify parameter into interpretable module groups.

    Why this exists:
        Group-level aggregation (attn/MLP/norm/embedding) makes it easy to see
        whether importance concentration is dominated by a specific subsystem.

    Args:
        parameter_name: Named parameter key.

    Returns:
        Module-group label string.
    """

    if parameter_name.startswith('model.embed_tokens.'):
        return 'embed_tokens'
    if parameter_name.startswith('lm_head.'):
        return 'lm_head'
    if parameter_name.startswith('model.norm.'):
        return 'final_norm'

    if '.self_attn.q_proj.' in parameter_name:
        return 'attn_q_proj'
    if '.self_attn.k_proj.' in parameter_name:
        return 'attn_k_proj'
    if '.self_attn.v_proj.' in parameter_name:
        return 'attn_v_proj'
    if '.self_attn.o_proj.' in parameter_name:
        return 'attn_o_proj'

    if '.mlp.gate_proj.' in parameter_name:
        return 'mlp_gate_proj'
    if '.mlp.up_proj.' in parameter_name:
        return 'mlp_up_proj'
    if '.mlp.down_proj.' in parameter_name:
        return 'mlp_down_proj'

    if '.input_layernorm.' in parameter_name:
        return 'input_layernorm'
    if '.post_attention_layernorm.' in parameter_name:
        return 'post_attention_layernorm'

    return 'other'


def format_layer_label(layer_index: int) -> str:
    """Format integer layer index into compact display label."""

    if layer_index < 0:
        return 'non_transformer'
    return f"L{layer_index:02d}"


def collect_parameter_mass_dataframe(
    importance_tensors: Mapping[str, torch.Tensor],
) -> Tuple[pd.DataFrame, float]:
    """Build per-parameter importance mass table.

    Args:
        importance_tensors: Importance tensors keyed by parameter name.

    Returns:
        Tuple of:
        - DataFrame with per-parameter metadata and mass statistics.
        - Total importance mass across all parameters.
    """

    rows = []
    total_mass = 0.0

    for parameter_name, importance_tensor in importance_tensors.items():
        # Absolute value is used defensively against tiny numerical negatives.
        importance_abs = importance_tensor.detach().to(torch.float32).abs()

        parameter_mass = float(importance_abs.sum().item())
        parameter_numel = int(importance_abs.numel())

        layer_index = extract_layer_index(parameter_name)
        normalized_layer_index = int(layer_index) if layer_index is not None else -1

        rows.append(
            {
                'parameter_name': parameter_name,
                'layer_index': normalized_layer_index,
                'layer_label': format_layer_label(normalized_layer_index),
                'module_group': classify_module_group(parameter_name),
                'numel': parameter_numel,
                'mass': parameter_mass,
                'mean': float(parameter_mass / max(parameter_numel, 1)),
                'max': float(importance_abs.max().item()) if parameter_numel > 0 else 0.0,
            }
        )

        total_mass += parameter_mass

    parameter_df = pd.DataFrame(rows)
    if parameter_df.empty:
        return parameter_df, total_mass

    if total_mass > 0.0:
        parameter_df['mass_share_global'] = parameter_df['mass'] / total_mass
    else:
        parameter_df['mass_share_global'] = 0.0

    return parameter_df, total_mass


def build_layer_module_summaries(
    parameter_df: pd.DataFrame,
    total_mass: float,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Aggregate parameter importance mass to layer and layer-module levels.

    Args:
        parameter_df: Per-parameter importance mass table.
        total_mass: Total importance mass across all parameters.

    Returns:
        Tuple of:
        - Layer-level summary table.
        - Layer+module-group summary table.
    """

    if parameter_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    layer_df = (
        parameter_df
        .groupby(['layer_index', 'layer_label'], as_index=False)
        .agg(mass=('mass', 'sum'), numel=('numel', 'sum'))
    )
    layer_df['mass_share_global'] = 0.0 if total_mass <= 0.0 else (layer_df['mass'] / total_mass)
    layer_df['mean'] = layer_df['mass'] / layer_df['numel'].clip(lower=1)
    layer_df = layer_df.sort_values('mass_share_global', ascending=False).reset_index(drop=True)

    layer_module_df = (
        parameter_df
        .groupby(['layer_index', 'layer_label', 'module_group'], as_index=False)
        .agg(mass=('mass', 'sum'), numel=('numel', 'sum'))
    )
    layer_module_df['mass_share_global'] = (
        0.0 if total_mass <= 0.0 else (layer_module_df['mass'] / total_mass)
    )
    layer_module_df['mean'] = layer_module_df['mass'] / layer_module_df['numel'].clip(lower=1)
    layer_module_df = layer_module_df.sort_values('mass_share_global', ascending=False).reset_index(drop=True)

    return layer_df, layer_module_df


def resolve_num_attention_heads(reference_model_path: Path) -> Optional[int]:
    """Read number of attention heads from model config.

    Args:
        reference_model_path: Local model path with Hugging Face config.

    Returns:
        `num_attention_heads` when available, otherwise `None`.
    """

    try:
        config = AutoConfig.from_pretrained(str(reference_model_path), trust_remote_code=True)
    except Exception as exc:
        print(f"[warn] Failed to load AutoConfig from {reference_model_path}: {exc}")
        return None

    num_heads = getattr(config, 'num_attention_heads', None)
    if num_heads is None:
        print('[warn] num_attention_heads is missing in model config; head analysis is skipped.')
        return None

    return int(num_heads)


def collect_attention_head_mass(
    importance_tensors: Mapping[str, torch.Tensor],
    num_attention_heads: int,
    total_mass: float,
) -> pd.DataFrame:
    """Compute per-layer per-head importance mass from Q/K/V projection weights.

    Important note:
        This decomposition assumes Q/K/V projection output channels are grouped
        contiguously by head, matching standard transformer implementations.

    Args:
        importance_tensors: Importance tensors keyed by parameter name.
        num_attention_heads: Number of attention heads from model config.
        total_mass: Total importance mass across all parameters.

    Returns:
        DataFrame with per-head importance mass and global mass share.
    """

    rows = []

    for parameter_name, importance_tensor in importance_tensors.items():
        match = ATTN_QKV_WEIGHT_PATTERN.match(parameter_name)
        if match is None:
            continue

        layer_index = int(match.group(1))
        projection_name = str(match.group(2))

        importance_abs = importance_tensor.detach().to(torch.float32).abs()
        if importance_abs.ndim != 2:
            # Q/K/V projection weights are expected to be 2D matrices.
            continue

        out_features = int(importance_abs.shape[0])
        in_features = int(importance_abs.shape[1])
        if out_features % int(num_attention_heads) != 0:
            # Skip if head partitioning is ambiguous.
            continue

        head_dim = out_features // int(num_attention_heads)
        per_head_mass = importance_abs.reshape(num_attention_heads, head_dim, in_features).sum(dim=(1, 2))

        for head_index, mass_value in enumerate(per_head_mass.tolist()):
            rows.append(
                {
                    'layer_index': layer_index,
                    'layer_label': format_layer_label(layer_index),
                    'projection': projection_name,
                    'head_index': int(head_index),
                    'mass': float(mass_value),
                }
            )

    head_projection_df = pd.DataFrame(rows)
    if head_projection_df.empty:
        return head_projection_df

    # Aggregate q/k/v into one row per (layer, head).
    head_df = (
        head_projection_df
        .groupby(['layer_index', 'layer_label', 'head_index'], as_index=False)
        .agg(mass=('mass', 'sum'))
    )

    attention_total_mass = float(head_df['mass'].sum())
    head_df['global_mass_share'] = 0.0 if total_mass <= 0.0 else (head_df['mass'] / total_mass)
    head_df['attention_mass_share'] = (
        0.0 if attention_total_mass <= 0.0 else (head_df['mass'] / attention_total_mass)
    )

    head_df = head_df.sort_values('global_mass_share', ascending=False).reset_index(drop=True)
    return head_df


def resolve_importance_paths_for_dominance(runtime_output_root: Path) -> Dict[str, Path]:
    """Resolve importance paths for dominance diagnostics.

    Args:
        runtime_output_root: Runtime root for metadata fallback.

    Returns:
        Mapping `task_name -> importance_path`.

    Raises:
        RuntimeError: If no valid mapping can be resolved.
    """

    if 'importance_paths' in globals():
        candidate_paths = globals().get('importance_paths')
        if isinstance(candidate_paths, Mapping) and len(candidate_paths) > 0:
            return {str(task): Path(str(path)) for task, path in candidate_paths.items()}

    summary_path = runtime_output_root / 'metadata' / 'jwcm_v2_run_summary.json'
    if summary_path.exists():
        summary_payload = json.loads(summary_path.read_text(encoding='utf-8'))
        summary_paths = summary_payload.get('importance_paths', {})
        if isinstance(summary_paths, Mapping) and len(summary_paths) > 0:
            return {str(task): Path(str(path)) for task, path in summary_paths.items()}

    raise RuntimeError(
        'Unable to resolve importance paths for dominance diagnostics. '
        'Run attribution stage first, or verify jwcm_v2_run_summary.json.'
    )


TOPK_LAYER_ROWS = 20
TOPK_LAYER_MODULE_ROWS = 30
TOPK_HEAD_ROWS = 40

importance_paths_by_task = resolve_importance_paths_for_dominance(runtime_output_root=RUNTIME.output_root)

# Use first task model config as attention-head reference.
reference_model_path = Path(TASK_SPECS[0].model_path)
num_attention_heads = resolve_num_attention_heads(reference_model_path=reference_model_path)
print(f"Attention head count from config: {num_attention_heads}")

for task_spec in TASK_SPECS:
    task_name = task_spec.name
    if task_name not in importance_paths_by_task:
        print(f"[warn] Missing importance path for task={task_name}; skipping.")
        continue

    importance_path = Path(importance_paths_by_task[task_name])

    print(f"\n=== JWCM dominant-parameter diagnostics: task={task_name} ===")

    importance_tensors = load_importance_dictionary_for_task(
        task_name=task_name,
        importance_path=importance_path,
    )
    parameter_df, total_mass = collect_parameter_mass_dataframe(
        importance_tensors=importance_tensors,
    )

    if parameter_df.empty:
        print(f"No importance tensors found for task={task_name}; skipping.")
        continue

    layer_df, layer_module_df = build_layer_module_summaries(
        parameter_df=parameter_df,
        total_mass=total_mass,
    )

    print(f"Total importance mass: {total_mass:.6e}")
    print(f"Num parameter tensors: {len(parameter_df)}")

    print("\nTop layers by global importance mass share:")
    display(layer_df.head(TOPK_LAYER_ROWS)[['layer_label', 'mass_share_global', 'mass', 'numel', 'mean']])

    print("\nTop (layer, module_group) by global importance mass share:")
    display(
        layer_module_df.head(TOPK_LAYER_MODULE_ROWS)[
            ['layer_label', 'module_group', 'mass_share_global', 'mass', 'numel', 'mean']
        ]
    )

    print("\nTop parameter tensors by global importance mass share:")
    display(
        parameter_df
        .sort_values('mass_share_global', ascending=False)
        .head(20)[['parameter_name', 'layer_label', 'module_group', 'mass_share_global', 'mean', 'max']]
        .reset_index(drop=True)
    )

    head_df = pd.DataFrame()
    if num_attention_heads is not None and num_attention_heads > 0:
        head_df = collect_attention_head_mass(
            importance_tensors=importance_tensors,
            num_attention_heads=num_attention_heads,
            total_mass=total_mass,
        )

    if not head_df.empty:
        print("\nTop attention heads by global importance mass share (aggregated over q/k/v):")
        display(
            head_df.head(TOPK_HEAD_ROWS)[
                ['layer_label', 'head_index', 'global_mass_share', 'attention_mass_share', 'mass']
            ]
        )
    else:
        print("\nAttention head analysis unavailable (config missing or no q/k/v matrices matched).")

    # Visualization panel: top layers + layer-module heatmap + head heatmap.
    figure, axes = plt.subplots(1, 3, figsize=(24, 6))

    # Panel 1: top layers bar chart.
    top_layer_plot_df = layer_df.head(TOPK_LAYER_ROWS).sort_values('mass_share_global', ascending=True)
    axes[0].barh(top_layer_plot_df['layer_label'], top_layer_plot_df['mass_share_global'])
    axes[0].set_title(f"Top layers by importance mass share ({task_name})")
    axes[0].set_xlabel('Global importance mass share')
    axes[0].set_ylabel('Layer')

    # Panel 2: layer x module-group heatmap.
    layer_module_pivot = (
        layer_module_df
        .pivot(index='layer_label', columns='module_group', values='mass_share_global')
        .fillna(0.0)
    )

    # Sort layer labels numerically, keeping non-transformer at the end.
    def _layer_sort_key(label: str) -> Tuple[int, int]:
        if label == 'non_transformer':
            return (9999, 9999)
        if label.startswith('L') and label[1:].isdigit():
            return (0, int(label[1:]))
        return (9998, 9998)

    sorted_layer_labels = sorted(layer_module_pivot.index.tolist(), key=_layer_sort_key)
    layer_module_pivot = layer_module_pivot.loc[sorted_layer_labels]

    heatmap_module = axes[1].imshow(layer_module_pivot.to_numpy(), aspect='auto', cmap='magma')
    axes[1].set_title(f"Layer x module importance mass share ({task_name})")
    axes[1].set_xlabel('Module group')
    axes[1].set_ylabel('Layer')
    axes[1].set_xticks(range(len(layer_module_pivot.columns)))
    axes[1].set_xticklabels(layer_module_pivot.columns.tolist(), rotation=45, ha='right')
    axes[1].set_yticks(range(len(layer_module_pivot.index)))
    axes[1].set_yticklabels(layer_module_pivot.index.tolist())
    figure.colorbar(heatmap_module, ax=axes[1], fraction=0.046, pad=0.04)

    # Panel 3: layer x head heatmap for q/k/v projections.
    if not head_df.empty:
        head_pivot = (
            head_df
            .pivot(index='layer_label', columns='head_index', values='global_mass_share')
            .fillna(0.0)
        )
        sorted_head_layers = sorted(head_pivot.index.tolist(), key=_layer_sort_key)
        head_pivot = head_pivot.loc[sorted_head_layers]

        heatmap_head = axes[2].imshow(head_pivot.to_numpy(), aspect='auto', cmap='viridis')
        axes[2].set_title(f"Attention head importance mass share ({task_name})")
        axes[2].set_xlabel('Head index')
        axes[2].set_ylabel('Layer')
        axes[2].set_xticks(range(len(head_pivot.columns)))
        axes[2].set_xticklabels(head_pivot.columns.tolist())
        axes[2].set_yticks(range(len(head_pivot.index)))
        axes[2].set_yticklabels(head_pivot.index.tolist())
        figure.colorbar(heatmap_head, ax=axes[2], fraction=0.046, pad=0.04)
    else:
        axes[2].axis('off')
        axes[2].text(
            0.5,
            0.5,
            'Head-level analysis unavailable',
            ha='center',
            va='center',
            fontsize=12,
        )

    plt.tight_layout()
    plt.show()


## Notes

- `AttributionConfig.mode='exact_token_abs'` is formula-faithful but expensive.
- If runtime is too high, switch to `sequence_sum_approx` and/or reduce `max_backprop_tokens`.
- You can tune sparsification aggressiveness via:
  - `MERGE_CFG.sparsify_mode='absolute'` + `sparsify_kappa`
  - `MERGE_CFG.sparsify_mode='sampled_quantile'` + `sparsify_quantile`

### Token-Delta Source Priority

This notebook tries token-delta sources in this order:

1. Fisher-pipeline `correct_rollout_trajectories.parquet` reuse.
2. Legacy JWCM precompute fallback (`all_token_deltas.csv` + `sequence_cache.pt`).
3. Online generation fallback from Fisher validation prompts.

### Optional Legacy Precompute Command (Fallback Only)

```bash
torchrun --standalone --nproc_per_node=8 scripts/run_collect_token_deltas_8gpu.py   --validation-root /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/validation   --output-root /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-jwcm-v2/distributed_token_precompute   --dtype bf16
```

- `ATTRIBUTION_VALIDATION_SAMPLES_PER_TASK=None` keeps all Fisher validation prompts for alignment.
- `ATTRIBUTION_CORRECT_ROLLOUT_SAMPLES_PER_TASK=128` samples 128 correct rollout rows per task (trajectory-level).
- If Fisher rollout coverage is partial (for example, math only subset), fallback sources automatically fill missing paths.


In [ ]:
# IF top-k sparsification checkpoints from IF-only importance
# This cell builds 3 IF-only sparse task-vector checkpoints where only the global
# top {1%, 5%, 10%} IF importance coordinates are allowed to update the base model.
# Evaluation is intentionally not run here; only checkpoint artifacts are saved.

import gc
import math
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Tuple

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM


def resolve_if_importance_path(runtime_output_root: Path, task_name: str = "if") -> Path:
    """Resolve the saved importance tensor path for a given task.

    Why this helper exists:
        Notebook execution is often partial/restart-based, so in-memory variables
        such as `importance_paths` may not exist when this cell is rerun alone.
        This helper first checks in-memory state, then falls back to run-summary metadata.

    Args:
        runtime_output_root: Runtime output directory that contains metadata JSON.
        task_name: Task key to resolve, defaulting to `if`.

    Returns:
        Absolute `Path` to the serialized task importance dictionary (`.pt`).

    Raises:
        KeyError: If no importance path can be resolved for `task_name`.
        FileNotFoundError: If the resolved path does not exist on disk.
    """

    # Priority 1: in-memory map from the earlier attribution/merge stage.
    importance_map = globals().get("importance_paths")
    if isinstance(importance_map, dict) and task_name in importance_map:
        resolved = Path(importance_map[task_name])
        if resolved.exists():
            return resolved

    # Priority 2: persisted run summary (works after kernel restarts).
    summary_path = runtime_output_root / "metadata" / "jwcm_v2_run_summary.json"
    if not summary_path.exists():
        raise KeyError(
            f"Could not resolve importance path for task '{task_name}'. "
            f"Missing in-memory `importance_paths` and metadata file: {summary_path}"
        )

    summary_payload = load_json(summary_path)
    summary_importance_map = summary_payload.get("importance_paths", {})
    if task_name not in summary_importance_map:
        raise KeyError(
            f"Task '{task_name}' not found in run summary importance paths: {summary_path}"
        )

    resolved = Path(summary_importance_map[task_name])
    if not resolved.exists():
        raise FileNotFoundError(f"Resolved importance path does not exist: {resolved}")

    return resolved


def compute_global_topk_threshold_plan(
    importance_dict: Mapping[str, torch.Tensor],
    parameter_names: List[str],
    keep_ratios: Iterable[float],
) -> Tuple[Dict[float, Dict[str, Any]], int]:
    """Compute exact global top-k thresholds and counts for multiple keep ratios.

    Method:
        1. Flatten and concatenate all selected IF importance tensors.
        2. For each ratio `r`, compute `k = ceil(r * N)` with `N` total coordinates.
        3. Use `torch.kthvalue` to obtain the exact threshold value for top-k selection.
        4. Record strict-above and equal-to-threshold counts for deterministic tie handling.

    Args:
        importance_dict: Mapping `parameter_name -> IF importance tensor`.
        parameter_names: Ordered parameter names to include in global top-k ranking.
        keep_ratios: Ratios in `(0, 1]`, e.g., `[0.01, 0.05, 0.10]`.

    Returns:
        Tuple of:
            - plan mapping `ratio -> {target_coords, threshold, strict_count, equal_count}`
            - total number of ranked coordinates

    Raises:
        ValueError: If `parameter_names` is empty or ratios are out of range.
    """

    if not parameter_names:
        raise ValueError("No parameters were provided for top-k threshold computation.")

    normalized_ratios: List[float] = []
    for ratio in keep_ratios:
        ratio_value = float(ratio)
        if ratio_value <= 0.0 or ratio_value > 1.0:
            raise ValueError(f"keep_ratio must be in (0, 1], got {ratio_value}.")
        normalized_ratios.append(ratio_value)

    # Concatenate once so thresholds for all requested ratios can share the same pool.
    flattened_importance = torch.cat(
        [importance_dict[name].reshape(-1).to(torch.float32) for name in parameter_names],
        dim=0,
    )
    total_coordinates = int(flattened_importance.numel())

    threshold_plan: Dict[float, Dict[str, Any]] = {}
    for ratio in normalized_ratios:
        target_coordinates = max(1, int(math.ceil(total_coordinates * ratio)))

        # kth is 1-indexed: top-k largest == (N-k+1)-th smallest threshold.
        kth_index = total_coordinates - target_coordinates + 1
        threshold_value = float(torch.kthvalue(flattened_importance, kth_index).values.item())

        strict_count = 0
        equal_count = 0
        for name in parameter_names:
            tensor = importance_dict[name].to(torch.float32)
            strict_count += int((tensor > threshold_value).sum().item())
            equal_count += int((tensor == threshold_value).sum().item())

        threshold_plan[ratio] = {
            "target_coordinates": int(target_coordinates),
            "threshold": float(threshold_value),
            "strict_count": int(strict_count),
            "equal_count": int(equal_count),
        }

    # Free concatenated buffer after thresholds are extracted.
    del flattened_importance
    gc.collect()

    return threshold_plan, total_coordinates


def apply_if_topk_delta_update_inplace(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    if_importance: Mapping[str, torch.Tensor],
    parameter_names: List[str],
    threshold: float,
    target_coordinates: int,
    strict_count: int,
) -> Dict[str, Any]:
    """Apply IF task-vector update to `base_model` on only top-k IF-importance coordinates.

    Update rule per coordinate:
        `theta_base <- theta_base + (theta_if - theta_base) * mask`
    where `mask=1` for selected top-k IF-importance coordinates and `0` otherwise.

    Tie handling:
        When many coordinates equal the threshold, this function fills the remaining
        tie budget deterministically in parameter iteration order and flat-index order.
        This guarantees exactly `target_coordinates` selected coordinates.

    Args:
        base_model: Base anchor model to be updated in-place.
        if_model: IF tuned model used to build task-vector deltas.
        if_importance: IF importance dictionary keyed by parameter name.
        parameter_names: Ordered parameter names included in top-k ranking.
        threshold: Global threshold corresponding to requested keep ratio.
        target_coordinates: Exact number of coordinates to keep.
        strict_count: Number of coordinates strictly above threshold.

    Returns:
        Summary dictionary with effective selected counts and tie usage.

    Raises:
        RuntimeError: If exact target coordinate count is not achieved.
    """

    base_named_parameters = dict(base_model.named_parameters())
    if_named_parameters = dict(if_model.named_parameters())

    tie_budget = max(0, int(target_coordinates) - int(strict_count))
    initial_tie_budget = int(tie_budget)
    selected_coordinates = 0

    with torch.no_grad():
        for parameter_name in tqdm(parameter_names, desc="Apply IF top-k sparse update"):
            base_parameter = base_named_parameters[parameter_name]
            if_parameter = if_named_parameters[parameter_name]

            # Compute IF task-vector delta in float32 for stable masking math.
            base_tensor_fp32 = base_parameter.data.detach().to(torch.float32)
            if_tensor_fp32 = if_parameter.data.detach().to(torch.float32)
            delta_tensor = if_tensor_fp32 - base_tensor_fp32

            importance_tensor = if_importance[parameter_name].to(torch.float32)
            selection_mask = importance_tensor > float(threshold)

            # Deterministically fill threshold ties so selected count matches target exactly.
            if tie_budget > 0:
                equal_mask = importance_tensor == float(threshold)
                equal_count = int(equal_mask.sum().item())

                if equal_count <= tie_budget:
                    selection_mask = torch.logical_or(selection_mask, equal_mask)
                    tie_budget -= equal_count
                elif equal_count > 0:
                    equal_indices = torch.nonzero(equal_mask.reshape(-1), as_tuple=False).squeeze(1)
                    mask_flat = selection_mask.reshape(-1)
                    mask_flat[equal_indices[:tie_budget]] = True
                    selection_mask = mask_flat.view_as(selection_mask)
                    tie_budget = 0

            selected_coordinates += int(selection_mask.sum().item())

            # Apply sparse IF delta update only on selected coordinates.
            sparse_delta = delta_tensor * selection_mask.to(delta_tensor.dtype)
            updated_tensor = base_tensor_fp32 + sparse_delta
            base_parameter.data.copy_(updated_tensor.to(base_parameter.dtype))

    if selected_coordinates != int(target_coordinates):
        raise RuntimeError(
            "Top-k sparse IF update selected unexpected number of coordinates: "
            f"selected={selected_coordinates}, target={target_coordinates}"
        )

    return {
        "threshold": float(threshold),
        "target_coordinates": int(target_coordinates),
        "selected_coordinates": int(selected_coordinates),
        "selected_ratio": float(selected_coordinates / max(1, sum(if_importance[name].numel() for name in parameter_names))),
        "strict_count": int(strict_count),
        "tie_budget_used": int(initial_tie_budget),
        "tie_budget_remaining": int(tie_budget),
    }


# -----------------------------------------------------------------------------
# Main execution: generate and save IF sparse checkpoints for keep ratios 1/5/10%.
# -----------------------------------------------------------------------------
keep_ratios = [0.01, 0.05, 0.10]
if_task_spec = next(task for task in TASK_SPECS if task.name == "if")
if_importance_path = resolve_if_importance_path(RUNTIME.output_root, task_name="if")
if_importance = torch.load(if_importance_path, map_location="cpu")

# Load IF model once; base model is reloaded per ratio so every run starts from anchor.
if_model, _if_tokenizer = load_causal_lm(
    model_name_or_path=if_task_spec.model_path,
    torch_dtype=RUNTIME.model_dtype,
    device="cpu",
)

# Filter to floating-point parameter tensors that are present in the IF importance dict.
if_model_named = dict(if_model.named_parameters())
eligible_parameter_names: List[str] = [
    name
    for name, parameter in if_model_named.items()
    if torch.is_floating_point(parameter.data) and name in if_importance
]

if not eligible_parameter_names:
    raise ValueError("No eligible floating-point parameters found for IF top-k sparsification.")

threshold_plan, total_coordinates = compute_global_topk_threshold_plan(
    importance_dict=if_importance,
    parameter_names=eligible_parameter_names,
    keep_ratios=keep_ratios,
)

sparse_if_root = RUNTIME.output_root / "if_importance_topk_sparsified"
sparse_if_root.mkdir(parents=True, exist_ok=True)

sparse_if_summary: Dict[str, Any] = {
    "created_at": now_iso(),
    "method": "if_importance_topk_sparse_delta_update",
    "runtime": {
        "base_model_id": RUNTIME.base_model_id,
        "output_root": str(RUNTIME.output_root),
        "seed": RUNTIME.seed,
        "model_dtype": str(RUNTIME.model_dtype),
        "device": RUNTIME.device,
    },
    "if_model_path": str(if_task_spec.model_path),
    "if_importance_path": str(if_importance_path),
    "total_coordinates": int(total_coordinates),
    "results": {},
}

for keep_ratio in keep_ratios:
    ratio_plan = threshold_plan[float(keep_ratio)]

    # Reload base anchor so each sparsity experiment is isolated and comparable.
    ratio_base_model, ratio_tokenizer = load_causal_lm(
        model_name_or_path=RUNTIME.base_model_id,
        torch_dtype=RUNTIME.model_dtype,
        device="cpu",
    )
    validate_parameter_compatibility(ratio_base_model, if_model)

    update_summary = apply_if_topk_delta_update_inplace(
        base_model=ratio_base_model,
        if_model=if_model,
        if_importance=if_importance,
        parameter_names=eligible_parameter_names,
        threshold=float(ratio_plan["threshold"]),
        target_coordinates=int(ratio_plan["target_coordinates"]),
        strict_count=int(ratio_plan["strict_count"]),
    )

    ratio_percent = int(round(float(keep_ratio) * 100))
    ratio_output_dir = sparse_if_root / f"if_top_{ratio_percent}pct"

    save_merged_artifacts(
        model=ratio_base_model,
        tokenizer=ratio_tokenizer,
        output_dir=ratio_output_dir,
        metadata={
            "created_at": now_iso(),
            "method": "if_importance_topk_sparse_delta_update",
            "task": "if",
            "keep_ratio": float(keep_ratio),
            "keep_percent": int(ratio_percent),
            "runtime": {
                "base_model_id": RUNTIME.base_model_id,
                "seed": RUNTIME.seed,
                "model_dtype": str(RUNTIME.model_dtype),
                "device": RUNTIME.device,
            },
            "if_model_path": str(if_task_spec.model_path),
            "if_importance_path": str(if_importance_path),
            "threshold_plan": ratio_plan,
            "update_summary": update_summary,
        },
    )

    sparse_if_summary["results"][f"top_{ratio_percent}pct"] = {
        "keep_ratio": float(keep_ratio),
        "output_dir": str(ratio_output_dir),
        "threshold_plan": ratio_plan,
        "update_summary": update_summary,
    }

    print(
        f"Saved IF top-{ratio_percent}% sparse checkpoint: {ratio_output_dir} "
        f"(selected={update_summary['selected_coordinates']:,} / {total_coordinates:,})"
    )

    # Release ratio-specific model objects before next ratio.
    del ratio_base_model
    del ratio_tokenizer
    gc.collect()

# Persist one summary JSON for quick downstream evaluation orchestration.
summary_output_path = RUNTIME.output_root / "metadata" / "if_topk_sparsification_summary.json"
save_json(sparse_if_summary, summary_output_path)
print(f"Saved IF top-k sparsification summary: {summary_output_path}")

# Cleanup long-lived objects to keep notebook memory stable.
del _if_tokenizer
del if_model
del if_model_named
del if_importance
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


